# 4. 灰度变换、直方图与图像复原

**目标**：掌握图像像素域的增强方法（灰度变换、直方图处理）以及基础的图像去噪与复原技术，为后续的形态学、边缘检测和高级分割打下基础。

在上一章中，我们处理图像时主要关注**色彩空间**（把图像从 BGR 转到 HSV、YCrCb、Lab）和**颜色信息**的利用。
但现实中的图像往往存在**对比度不足、亮度偏暗、噪声干扰、模糊**等问题。本章要做的事情可以概括为一句话：**在像素域直接对灰度值进行映射和统计，从而增强图像、去除噪声、恢复模糊**。

这两类操作贯穿了几乎所有传统图像处理任务：医学影像增强、指纹识别、老照片修复、监控视频去噪，都建立在它们之上。本章会先把每一类操作的**原理**讲清楚，再用 OpenCV 函数验证。

> **代码独立性说明**：本章每个代码块都**完全自包含**——自己导入库、自己读取图像（或自建测试数据），不依赖其他任何小节。每个代码块都可以单独运行。

## 4.0 术语与变量名约定（先读这一段）

本章会频繁出现 **灰度变换、直方图、均衡化、去噪、复原** 这些**图像处理术语**。为了避免「术语」和「代码变量」混淆，本章统一约定：

| 图像处理术语 | 代码里的图像变量 | 说明 |
|--------------|------------------|------|
| 原始灰度图 | `img_gray` | 单通道灰度图 |
| 变换后图像 | `img_transformed` | 经过灰度变换后的图像 |
| 直方图 | `hist` | 一维数组，长度为 256 |
| 均衡化后图像 | `img_eq` | 经过直方图均衡化的图像 |
| 去噪后图像 | `img_denoised` | 经过滤波去噪后的图像 |
| 复原后图像 | `img_restored` | 经过维纳滤波或逆滤波后的图像 |

**两条硬规则**：

1. **不要把 `gray`、`hist`、`eq` 当变量名**——它们是图像处理术语，直接当变量名会让读者分不清「这是在说概念，还是在说某个数组」。统一用 `img_gray`、`hist`、`img_eq`。
2. **`img_gray` 只是一个普通的 NumPy 二维数组**，形状为 `(height, width)`，数值范围通常是 0~255。它不是特殊类型，OpenCV 也不会因为变量名叫 `img_gray` 就自动按灰度处理。

**本章所需的测试图像**

本章使用以下图像（存放在 `P&V/` 目录下）：

- `bone.jpg`：骨骼 X 光图，适合演示灰度变换、Gamma 校正、直方图均衡化、CLAHE；
- `fingerprint.png`：指纹图（干净图），适合演示反转变换、比特平面、灰度直方图；去噪、去模糊部分会在代码中**自行加噪/加模糊**再处理；
- `coin.jpg`：硬币图，适合演示灰度直方图、直方图匹配、图像修复；
- `grayLena.bmp`：灰度自然图像，用于和硬币图、指纹图对比直方图形态；
- `darkcolorful.png`：暗调彩色图（人像 + 草地 + 野餐布），颜色丰富、整体偏暗，适合演示彩色图像均衡化策略；
- `house.jpg`：故宫彩色图，颜色丰富、亮度正常，适合对比“保色策略”；
- `portraitGrass.png`：人像 + 绿色草地，颜色丰富、目标与背景差异明显，适合演示 2D 直方图与反向投影；
- 4.4 节会使用**自己合成的测试图**，因为自制图的灰度分区更明确，能更清楚地展示对比度拉伸和灰度级分层的效果。

主要内容：
1. 图像反转变换
2. 线性灰度变换
3. 非线性灰度变换：对数变换、幂律变换（Gamma 校正）
4. Gamma 校正：物理意义、LUT 实现
5. 分段线性变换：对比度拉伸、灰度级分层
6. 灰度变换之比特平面
7. 基于灰度变换调整图像色阶
8. 对比度与亮度调整：convertScaleAbs
9. 图像的灰度直方图：计算、绘制
10. 直方图均衡化：equalizeHist
11. 直方图匹配：calcHist + LUT 实现
12. 基于局部直方图统计量的图像增强
13. 限制对比度自适应直方图均衡化：createCLAHE
14. 彩色图像均衡化策略（YCrCb 的 Y / Lab 的 L 通道）
15. 2D 直方图：H-S 平面联合直方图
16. 直方图反向投影：calcBackProject
17. 噪声模型：高斯、椒盐、泊松、乘性噪声
18. 去噪基础：非局部均值、快速非局部均值、双边滤波
19. 去噪效果评估：PSNR、SSIM
20. 去模糊基础：维纳滤波、逆滤波
21. 图像修复：inpaint、INPAINT_TELEA、INPAINT_NS
22. 常见坑与附录

---
## 4.1 图像反转变换 (Image Negative)

反转变换将图像的灰度值进行反转，公式为 $s = L - 1 - r$，其中 $L$ 是灰度级数（通常为 256），$r$ 是原始灰度值，$s$ 是变换后的灰度值。

**应用场景**：常用于增强暗区域中的白色或灰色细节，例如医学影像中的数字减影血管造影（DSA），或者指纹图像中突出脊线。

**OpenCV 实现**：

```python
img_neg = 255 - img_gray
```

或者使用 `cv2.bitwise_not`：

```python
img_neg = cv2.bitwise_not(img_gray)
```

> **常见坑**：
> - **反转后图像全黑/全白**：如果误把 `255 - img_gray` 写成 `img_gray - 255`，uint8 会回绕，结果完全错误。解决：严格按 $s = 255 - r$ 写，或直接用 `cv2.bitwise_not`。
> - **对彩色图直接做 `255 - img_bgr`**：这是对 B/G/R 三个通道同时反转，会改变色调（相当于反色），不是“灰度反转”。如果只想反转亮度，应先转灰度或转 YCrCb 只处理 Y 通道。

In [ ]:
# ============ 4.1 实例：图像反转变换 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/fingerprint.png', cv2.IMREAD_GRAYSCALE)                  # 读取灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/fingerprint.png'        # 断言图像读取成功

# 反转变换
img_neg = 255 - img_gray                                                            # 公式 s = 255 - r

# 显示
plt.figure(figsize=(12, 5))                                                         # 画布

plt.subplot(1, 2, 1)                                                                # 原图
plt.imshow(img_gray, cmap='gray')                                                   # 显示
plt.title('Original (Fingerprint)')                                                 # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 2, 2)                                                                # 反转变换
plt.imshow(img_neg, cmap='gray')                                                    # 显示
plt.title('Negative Transform')                                                     # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：反转变换将亮暗互换，常用于增强暗部细节')                                # 打印结论

---
## 4.2 线性灰度变换 (Linear Gray-level Transformation)

线性变换公式：$s = a \cdot r + b$。

- $a$：对比度系数（$a > 1$ 增加对比度，$0 < a < 1$ 降低对比度）；
- $b$：亮度偏移（$b > 0$ 增加亮度，$b < 0$ 降低亮度）。

**超出 0~255 怎么办**：不需要自己手工写截断逻辑，OpenCV 的 `cv2.convertScaleAbs` 已经帮你处理了。

```python
img_linear = cv2.convertScaleAbs(img_gray, alpha=a, beta=b)
```

它的内部流程是：

1. 先做线性变换 $s = a \cdot r + b$；
2. 对结果取**绝对值** `|s|`；
3. 再**饱和截断**到 `uint8` 的范围 `[0, 255]`，也就是超过 255 的按 255 处理、低于 0 的按 0 处理；
4. 返回 `uint8` 图像。

所以直接调用 `cv2.convertScaleAbs` 即可，**不需要**再手动 `np.clip`。这也是本章 4.2 实例里用的方式：

```python
img_linear = cv2.convertScaleAbs(img_gray, alpha=1.5, beta=30)
```

**什么时候才需要手工用 NumPy**：

- 你想自己控制截断方式（比如不取绝对值，只做 `clip`，保留负值的处理逻辑）；
- 你要做的是更复杂的逐像素映射，OpenCV 没有现成函数；
- 你想在 `float32` 域里连续做多步运算，最后再统一截断回 `uint8`。

这时才写成：

```python
img_linear = np.clip(a * img_gray.astype(np.float32) + b, 0, 255).astype(np.uint8)
```

**注意 `convertScaleAbs` 的“取绝对值”语义**：当 `a > 0` 时，`a*r + b` 一般不会为负（除非 `b` 很小且 `r` 很小），取绝对值与直接截断结果一致；但当 `a` 或 `b` 为负时，`convertScaleAbs` 会先把负值变成正值再截断，和 `np.clip` 的结果不同。做常规对比度/亮度调整时 `a > 0`，两者等价，可以放心用 `convertScaleAbs`。

> **常见坑**：
> - **灰度变换后图像全黑/全白**：变换系数过大或过小，超出 0~255 后没有截断，uint8 回绕。解决：用 `cv2.convertScaleAbs`（自动饱和截断），或手工 `np.clip(..., 0, 255).astype(np.uint8)`。
> - **手工写 `a * img_gray + b` 直接 `astype(np.uint8)`**：`img_gray` 是 uint8，`a * img_gray` 会先按 uint8 运算再回绕，负值和超过 255 的值都会出错。解决：先 `img_gray.astype(np.float32)`，再运算，最后 `np.clip` 后转 `uint8`。
> - **误以为 `convertScaleAbs` 等价于 `np.clip`**：`convertScaleAbs` 会先取绝对值再截断，当 `a` 或 `b` 为负时结果和 `np.clip` 不同。做常规对比度/亮度调整时 `a > 0`，两者等价；一旦涉及负系数，就要明确自己要的是“取绝对值”还是“截断到 0”。

In [ ]:
# ============ 4.2 实例：线性灰度变换 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/bone.jpg', cv2.IMREAD_GRAYSCALE)                         # 读取灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/bone.jpg'               # 断言图像读取成功

# 线性变换：a=1.5 (增加对比度), b=30 (增加亮度)
img_linear = cv2.convertScaleAbs(img_gray, alpha=1.5, beta=30)                      # 线性变换

# 显示
plt.figure(figsize=(12, 5))                                                         # 画布

plt.subplot(1, 2, 1)                                                                # 原图
plt.imshow(img_gray, cmap='gray')                                                   # 显示
plt.title('Original (Bone)')                                                        # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 2, 2)                                                                # 线性变换
plt.imshow(img_linear, cmap='gray')                                                 # 显示
plt.title('Linear Transform (a=1.5, b=30)')                                         # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：线性变换通过调整 alpha 和 beta 改变对比度和亮度')                        # 打印结论

---
## 4.3 非线性灰度变换：对数变换与幂律变换（Gamma 校正）

### 4.3.1 对数变换

公式：$s = c \cdot \log(1 + r)$。

- 用于扩展低灰度值（暗部）细节，压缩高灰度值（亮部）细节；
- 典型应用：傅里叶频谱的显示。

**OpenCV 实现**：

```python
img_norm = img_gray / 255.0
img_log = np.log1p(img_norm)
img_log = (img_log / np.max(img_log)) * 255
img_log = np.uint8(img_log)
```

> **常见坑**：
> - **直接对 uint8 做 `np.log`**：`img_gray` 是整数，`np.log(img_gray)` 会先把整数转浮点，但 `log(0)` 会得到 `-inf`，且未归一化时数值范围不可控。解决：先 `img_gray / 255.0` 归一化到 `[0,1]`，用 `np.log1p` 避免 `log(0)`，最后再拉伸回 `[0,255]`。
> - **拉伸回 255 时用了固定系数**：`c` 应取 `255 / np.max(img_log)`，而不是写死一个常数，否则不同图像结果不一致。

### 4.3.2 幂律变换（Gamma 校正）

公式：$s = c \cdot r^\gamma$。

- $\gamma < 1$：扩展暗部细节（类似对数变换）；
- $\gamma > 1$：扩展亮部细节；
- $\gamma = 1$：恒等变换。

**物理意义**：Gamma 校正常用于校正显示设备的非线性响应。人眼对亮度的感知近似对数关系，而显示器的输出与输入电压近似幂律关系（$\gamma \approx 2.2$）。通过 Gamma 校正可以补偿这种非线性，使图像在人眼看来亮度均匀。

**LUT 实现**：使用 `cv2.LUT` 可以极大地加速像素级映射操作。

```python
gamma = 0.5
table = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
img_gamma = cv2.LUT(img_gray, table)
```

> **注意**：这里 LUT 中的指数直接就是 `gamma`，与公式 $s = c \cdot r^\gamma$ 一致。因此 `gamma = 0.5` 对应 $r^{0.5}$，会提亮暗部；`gamma = 2.0` 对应 $r^{2}$，会压暗暗部。参数、注释、标题三者要保持一致，避免“写着提亮、实际却压暗”的矛盾。

**为什么 Gamma 校正比对数变换更灵活、可调？**

根本原因在于：**对数变换的曲线形状是固定的，而 Gamma 校正的曲线形状由一个可调参数 $\gamma$ 决定**。

1. **对数变换没有调节旋钮**。公式 $s = c \cdot \log(1 + r)$ 里的 $c$ 只是把结果拉伸回 $[0, 255]$ 的归一化系数，它不改变曲线形状。对数曲线永远是“先陡后平”的固定形状：暗部拉伸强、亮部压缩强。你只有“用”或“不用”两种选择，无法控制“到底提亮多少暗部”。

2. **Gamma 校正的 $\gamma$ 就是调节旋钮**。$\gamma$ 直接决定曲线形状：
   - $\gamma < 1$：曲线向上凸，暗部被拉伸、亮部被压缩，图像整体变亮；
   - $\gamma > 1$：曲线向下凹，暗部被压缩、亮部被拉伸，图像整体变暗；
   - $\gamma = 1$：恒等变换，图像不变。

   而且 $\gamma$ 可以取任意正实数（0.3、0.5、0.7、1.5、2.2……），因此你能**连续地调节提亮力度**，从“轻微提亮”到“强力提亮”得到一系列效果。

3. **从曲线形状看两者的关系**。对数变换的曲线形状，大致相当于 Gamma 校正里某个固定 $\gamma$（约 0.3~0.4）的效果，但它不能变；Gamma 校正则把“提亮暗部”这件事参数化了：$\gamma$ 越小，暗部提亮越强；$\gamma$ 越接近 1，效果越温和。

4. **实际使用中的意义**。在医学影像、老照片修复、监控图像增强等场景里，不同图像的暗部程度差别很大：有的只是稍微偏暗，用 $\gamma = 0.8$ 就够；有的暗部严重欠曝，需要 $\gamma = 0.4$ 甚至更小；有的需要压暗，用 $\gamma = 1.5$。如果只有对数变换，你只能接受它固定的提亮力度；而 Gamma 校正让你**用一个参数就覆盖从压暗到强提亮的全部范围**，所以更灵活、更可调，也是实际图像增强中更常用的原因。

**一句话总结**：对数变换的曲线形状是“死”的，Gamma 校正的曲线形状由 $\gamma$ 决定，$\gamma$ 可连续取值，因此能按需调节提亮力度。

> **常见坑**：
> - **Gamma 校正越调越暗**：LUT 中写了 `1/gamma`，或 `gamma` 取值与注释相反。解决：统一约定 LUT 中直接用 `gamma`，`gamma < 1` 提亮、`gamma > 1` 压暗，并让参数、注释、标题三者一致。
> - **LUT 构建时用了整数除法**：`i / 255` 在 Python 3 是浮点除，没问题；但如果写成 `i // 255`，结果全是 0 或 1，图像会变成二值图。解决：确认是 `/` 而不是 `//`。
> - **LUT 长度不是 256**：`cv2.LUT` 要求 LUT 长度与输入灰度级数一致（8 位图是 256）。如果写成 `np.arange(0, 255)`，长度 255，会报错或结果错位。解决：用 `np.arange(0, 256)`。

In [ ]:
# ============ 4.3 实例：对数变换与 Gamma 校正 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/bone.jpg', cv2.IMREAD_GRAYSCALE)                         # 读取灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/bone.jpg'               # 断言图像读取成功

# 1. 对数变换
img_norm = img_gray / 255.0                                                         # 归一化到 [0, 1]
img_log = np.log1p(img_norm)                                                        # log(1+r)
img_log = (img_log / np.max(img_log)) * 255                                         # 拉伸回 [0, 255]
img_log = np.uint8(img_log)                                                         # 转 uint8

# 2. Gamma 校正 (LUT 实现，直接用 gamma 作为指数)
gamma = 0.5                                                                         # 小于1，提亮暗部
table = np.array([((i / 255.0) ** gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")  # 构建 LUT
img_gamma = cv2.LUT(img_gray, table)                                                # 应用 LUT

# 显示
plt.figure(figsize=(15, 5))                                                         # 画布

plt.subplot(1, 3, 1)                                                                # 原图
plt.imshow(img_gray, cmap='gray')                                                   # 显示
plt.title('Original')                                                               # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 3, 2)                                                                # 对数变换
plt.imshow(img_log, cmap='gray')                                                    # 显示
plt.title('Log Transform')                                                          # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 3, 3)                                                                # Gamma 校正
plt.imshow(img_gamma, cmap='gray')                                                  # 显示
plt.title(f'Gamma Correction (gamma={gamma})')                                      # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：对数变换和 Gamma 校正都能扩展暗部细节；对数曲线固定，Gamma 的指数可调，因此更灵活')  # 打印结论

---
## 4.4 分段线性变换 (Piecewise Linear Transformation)

### 4.4.1 对比度拉伸 (Contrast Stretching)

将原始窄范围的灰度值拉伸到整个灰度范围。例如，原始图像灰度集中在 `[100, 150]`，可以线性映射到 `[0, 255]`，从而增强对比度。

**OpenCV 实现**：

```python
img_stretch = cv2.normalize(img_gray, None, 0, 255, cv2.NORM_MINMAX)
```

`cv2.normalize(..., NORM_MINMAX)` 的原理是：先找到图像中的最小灰度值 `min` 和最大灰度值 `max`，然后把 `[min, max]` 线性拉伸到 `[0, 255]`。

**关键点**：只有当原图灰度范围**没有铺满 `[0, 255]`** 时，拉伸才会产生明显效果。如果原图本身背景接近 0、高光接近 255（例如硬币图、逆光剪影图），`min ≈ 0`、`max ≈ 255`，归一化后映射关系几乎是恒等的，图像看起来和原图一样。

因此本节用一个**自制低对比度图**来演示：把灰度人为压在 `[100, 150]`，再拉伸到 `[0, 255]`，对比效果会非常直观。

> **常见坑**：
> - **对比度拉伸看不出效果**：原图灰度已铺满 `[0,255]`，`NORM_MINMAX` 映射接近恒等。解决：换灰度集中的低对比度图，或先人为压缩灰度再拉伸。
> - **对彩色图直接 `cv2.normalize`**：`NORM_MINMAX` 会对 B/G/R 三个通道分别归一化，可能改变色调。如果只想增强亮度对比度，应先转 YCrCb 只对 Y 通道做。

### 4.4.2 灰度级分层 (Gray-level Slicing)

灰度级分层的含义是：**按灰度值圈定一个范围，把这个范围内的像素单独挑出来高亮显示**。它不改变整幅图的亮度，只做**选择性突出**，常用于医学影像、遥感图像、工业检测中突出某段灰度对应的区域。

它有两种常见做法：

**1. 二值化式分层 (Binary Slicing)**：范围内置为白色 255，范围外全部置为黑色 0。

```python
img_slice = np.zeros_like(img_gray)
mask = (img_gray >= lower) & (img_gray <= upper)
img_slice[mask] = 255
```

结果是一张黑白二值图，作用是**把目标区域从背景中分离出来**，便于后续做形态学、连通域分析、轮廓提取。

**2. 保留背景式分层 (Keep-background Slicing)**：范围内置为白色 255，范围外**保留原灰度值**。

```python
img_slice = img_gray.copy()
mask = (img_gray >= lower) & (img_gray <= upper)
img_slice[mask] = 255
```

结果是灰度图，目标区域被高亮成白色，背景和其他区域仍然可见。作用是**在保留上下文的前提下，视觉上突出目标灰度区域**。

**为什么用自制图演示**：真实照片的灰度分布连续且复杂，选一段灰度做分层时，效果往往不够明显；自制图的灰度分区明确（例如背景 = 50、圆 = 120、矩形 = 200），选 `[100, 150]` 做分层时，目标区域会干净利落地被挑出来，两种分层方式的区别也一目了然。

> **常见坑**：
> - **灰度级分层不明显**：分层区间选得太宽，覆盖了图像大部分灰度。解决：缩窄区间、压暗背景，或改用自制灰度分区图演示。
> - **用 `==` 而不是范围判断**：真实图像灰度连续，`img_gray == 120` 只能挑到极少数像素。解决：用 `(img_gray >= lower) & (img_gray <= upper)`。
> - **对彩色图直接做灰度级分层**：`img_bgr >= lower` 会对三个通道同时比较，结果不是“按亮度分层”。解决：先转灰度，或转 HSV 用 V 通道。

In [ ]:
# ============ 4.4 实例：对比度拉伸与灰度级分层 ============
# 本节完全独立：自己导入库、自己合成测试图

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

# ============ Part 1: 对比度拉伸 ============
# 构造一个灰度集中在 [100, 150] 的低对比度测试图
h, w = 300, 400                                                                     # 图像尺寸
img_low = np.zeros((h, w), dtype=np.uint8)                                          # 初始化

for x in range(w):                                                                  # 水平渐变，从 100 到 150
    img_low[:, x] = int(100 + (x / (w - 1)) * 50)                                   # 灰度值

cv2.circle(img_low, (120, 150), 60, 150, -1)                                        # 稍亮的圆
cv2.rectangle(img_low, (250, 80), (360, 220), 100, -1)                              # 稍暗的矩形

img_stretch = cv2.normalize(img_low, None, 0, 255, cv2.NORM_MINMAX)                 # 对比度拉伸

# 计算直方图
hist_low = cv2.calcHist([img_low], [0], None, [256], [0, 256])                      # 原图直方图
hist_stretch = cv2.calcHist([img_stretch], [0], None, [256], [0, 256])              # 拉伸后直方图

# 显示
plt.figure(figsize=(14, 8))                                                         # 画布

plt.subplot(2, 2, 1)                                                                # 低对比度原图
plt.imshow(img_low, cmap='gray', vmin=0, vmax=255)                                  # 显示
plt.title(f'Low Contrast (min={img_low.min()}, max={img_low.max()})')               # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 2, 2)                                                                # 原图直方图
plt.plot(hist_low)                                                                  # 绘制
plt.title('Histogram of Low Contrast')                                              # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.subplot(2, 2, 3)                                                                # 拉伸后
plt.imshow(img_stretch, cmap='gray', vmin=0, vmax=255)                              # 显示
plt.title(f'Contrast Stretching (min={img_stretch.min()}, max={img_stretch.max()})')  # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 2, 4)                                                                # 拉伸后直方图
plt.plot(hist_stretch)                                                              # 绘制
plt.title('Histogram after Stretching')                                             # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('对比度拉伸：原图灰度范围', img_low.min(), '-', img_low.max(), '，拉伸后', img_stretch.min(), '-', img_stretch.max())  # 打印结论

# ============ Part 2: 灰度级分层 ============
# 构造一张灰度分区明确的测试图
img = np.full((h, w), 50, dtype=np.uint8)                                           # 背景：灰度 50（暗）
cv2.circle(img, (120, 150), 70, 120, -1)                                            # 中间圆：灰度 120（落在 [100,150]）
cv2.rectangle(img, (250, 80), (360, 220), 200, -1)                                  # 右下矩形：灰度 200（亮）
cv2.circle(img, (60, 250), 35, 90, -1)                                              # 左下小圆：灰度 90（区间外）

lower, upper = 100, 150                                                             # 分层区间

# 二值化式分层 (Binary Slicing)
img_slice_binary = np.zeros_like(img)                                               # 全黑
mask = (img >= lower) & (img <= upper)                                              # 区间掩码
img_slice_binary[mask] = 255                                                        # 区间内置白

# 保留背景式分层 (Keep-background Slicing)
img_slice_keep = img.copy()                                                         # 保留原灰度
img_slice_keep[mask] = 255                                                          # 区间内置白

# 显示
plt.figure(figsize=(16, 10))                                                        # 画布

plt.subplot(2, 2, 1)                                                                # 原图
plt.imshow(img, cmap='gray', vmin=0, vmax=255)                                      # 显示
plt.title('Original (bg=50, circle=120, rect=200, small circle=90)')                # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 2, 2)                                                                # 原图直方图
hist = cv2.calcHist([img], [0], None, [256], [0, 256])                              # 计算
plt.plot(hist)                                                                      # 绘制
plt.title('Histogram of Original')                                                  # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.subplot(2, 2, 3)                                                                # 二值化式分层
plt.imshow(img_slice_binary, cmap='gray', vmin=0, vmax=255)                         # 显示
plt.title(f'Binary Slicing [{lower}-{upper}] (inside=255, outside=0)')              # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 2, 4)                                                                # 保留背景式分层
plt.imshow(img_slice_keep, cmap='gray', vmin=0, vmax=255)                           # 显示
plt.title(f'Keep-background Slicing [{lower}-{upper}] (inside=255, outside=original)')  # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('灰度级分层：背景灰度 50，中间圆 120，右下矩形 200，左下小圆 90')             # 打印各区域灰度
print(f'分层区间: [{lower}, {upper}]')                                              # 打印分层区间
print('Binary Slicing：只保留区间内像素，其余置黑')                                  # 打印结论
print('Keep-background Slicing：区间内置白高亮，其余保留原灰度')                     # 打印结论

---
## 4.5 灰度变换之比特平面 (Bit-plane Slicing)

### 4.5.1 什么是比特平面

8 位灰度图的每个像素是 0~255 的整数，写成二进制正好是 8 位。例如灰度值 `180` 写成二进制是 `10110100`。把这 8 位**按位“竖着切”**：

- 所有像素的第 7 位拼成一张图，叫第 7 位平面；
- 所有像素的第 6 位拼成一张图，叫第 6 位平面；
- ……
- 所有像素的第 0 位拼成一张图，叫第 0 位平面。

一共得到 **8 张二值图**，每张只有 0 和 1 两种取值，这就是 8 个比特平面。

### 4.5.2 怎么提取第 $i$ 位

```python
bit_plane = (img_gray >> i) & 1
bit_plane = bit_plane * 255
```

- `img_gray >> i`：把每个像素右移 `i` 位，让第 `i` 位移到最低位；
- `& 1`：只保留最低位，其余位清零；
- `* 255`：把 0/1 映射到 0/255，方便显示（0 黑，1 白）。

### 4.5.3 每一位的权重不同

| 位 | 权重 | 该位为 1 时贡献的灰度值 |
|----|------|------------------------|
| 第 7 位（MSB） | 2⁷ = 128 | +128 |
| 第 6 位 | 2⁶ = 64 | +64 |
| 第 5 位 | 2⁵ = 32 | +32 |
| 第 4 位 | 2⁴ = 16 | +16 |
| 第 3 位 | 2³ = 8 | +8 |
| 第 2 位 | 2² = 4 | +4 |
| 第 1 位 | 2¹ = 2 | +2 |
| 第 0 位（LSB） | 2⁰ = 1 | +1 |

权重越大，对灰度值的贡献越大，越决定像素“大致是亮还是暗”。

### 4.5.4 每一位平面的取值规律

第 `i` 位平面的取值，按该位权重形成的周期翻转：

| 位 | 权重 | 翻转间隔 | 0~255 内翻转次数 | 黑白段数 |
|----|------|----------|------------------|----------|
| 第 7 位 | 128 | 128 | 1 | 2 |
| 第 6 位 | 64 | 64 | 3 | 4 |
| 第 5 位 | 32 | 32 | 7 | 8 |
| 第 4 位 | 16 | 16 | 15 | 16 |
| 第 3 位 | 8 | 8 | 31 | 32 |
| 第 2 位 | 4 | 4 | 63 | 64 |
| 第 1 位 | 2 | 2 | 127 | 128 |
| 第 0 位 | 1 | 1 | 255 | 256 |

几个关键点：

- **第 7 位平面**：等价于“灰度值是否 ≥ 128”的二值图。灰度 ≥128 为白，<128 为黑，只有两大块，直接对应“亮区/暗区”，所以包含主要视觉信息。
- **第 6 位平面**：**不是**简单的“是否 ≥ 64”，而是在 `[64,127]` 和 `[192,255]` 为白，在 `[0,63]` 和 `[128,191]` 为黑，共 4 段。
- **第 5 位平面**：比第 6 位翻转更多，在 `[32,63]`、`[96,127]`、`[160,191]`、`[224,255]` 为白，其余为黑，共 8 段。
- **越低位，翻转越频繁，黑白交替越密**：第 3、2、1、0 位的段数分别达到 32、64、128、256，视觉上越来越像随机噪点。

### 4.5.5 为什么高位包含视觉信息、低位是噪声

- **高位平面**：权重高（128、64、32……），决定像素“大致是亮还是暗”，构成图像的主要结构，所以包含大部分视觉信息；
- **低位平面**：权重低（4、2、1），只影响微小变化，容易被传感器噪声、量化误差、压缩误差主导，所以通常表现为噪声。

用一个类比：把灰度值想成“钱”，第 7 位是 128 元，第 6 位是 64 元……第 0 位是 1 元。128 元、64 元决定“买不买得起”，是主要信息；1 元、2 元的零头对总价影响很小，还容易随机变化，相当于噪声。

### 4.5.6 应用

- **图像压缩**：只保留高几位平面，丢掉低位平面，可以在视觉损失不大的前提下压缩数据；
- **图像增强**：把若干高位平面重新组合，或对特定比特平面做处理，可以增强某些结构；
- **特征提取**：低位平面中的随机性有时也被用于纹理分析或隐写检测。

### 4.5.7 一句话总结

比特平面就是把每个像素的 8 位二进制按位拆开，每一位单独组成一张二值图。高位平面权重大、翻转少、包含主要视觉信息；低位平面权重小、翻转频繁、通常表现为噪声。第 7 位平面等价于“灰度值是否 ≥128”，第 6 位及更低则按各自的权重周期翻转，越低位越密。

> **常见坑**：
> - **误以为第 6 位平面是“是否 ≥ 64”**：第 6 位在 0~255 内翻转多次，不是简单阈值。解决：记住第 `i` 位每 `2^i` 个灰度级翻转一次；第 7 位才等价于“是否 ≥128”。
> - **位移方向写反**：写成 `(img_gray << i) & 1`，结果只剩最低位，全是 0 或 1 的错位图。解决：用 `(img_gray >> i) & 1`。
> - **忘了 `* 255`**：`(img_gray >> i) & 1` 得到的是 0/1 二值图，直接 `imshow` 会几乎全黑。解决：`bit_plane = bit_plane * 255` 映射到 0/255。

In [ ]:
# ============ 4.5 实例：比特平面分层 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/fingerprint.png', cv2.IMREAD_GRAYSCALE)                  # 读取灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/fingerprint.png'        # 断言图像读取成功

plt.figure(figsize=(15, 8))                                                         # 画布
for i in range(8):                                                                  # 遍历 8 个比特平面
    bit_plane = (img_gray >> i) & 1                                                 # 提取第 i 位
    bit_plane = bit_plane * 255                                                     # 映射到 0-255
    
    plt.subplot(2, 4, i+1)                                                          # 子图
    plt.imshow(bit_plane, cmap='gray')                                              # 显示
    plt.title(f'Bit Plane {i} (weight={2**i})')                                     # 标题（含权重）
    plt.axis('off')                                                                 # 关闭坐标轴
plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：高位平面（7,6）权重大、翻转少，包含主要视觉信息；低位平面（0,1）权重小、翻转频繁，主要是噪声')  # 打印结论

---
## 4.6 图像的灰度直方图

直方图反映了图像中灰度值的分布情况。

**OpenCV 函数**：

```python
hist = cv2.calcHist([img_gray], [0], None, [256], [0, 256])
```

**Matplotlib 绘制**：

```python
plt.plot(hist)
```

**应用**：判断图像对比度、亮度，为直方图均衡化提供依据。

不同图像的直方图形态差别很大：硬币图背景暗、硬币亮，灰度分布会出现明显的峰；指纹图脊线和谷线交替，灰度分布更集中，直方图通常呈现单一尖峰；灰度 Lena 图是自然图像，灰度分布宽、多峰。下面用这三张图分别计算并绘制直方图，便于对比。

> **常见坑**：
> - **`cv2.calcHist` 返回的是二维数组**：形状 `(256, 1)`，直接 `plt.plot(hist)` 能画，但 `hist[i]` 是数组不是标量。解决：用 `hist = cv2.calcHist(...).ravel()` 转成一维。
> - **忘了指定通道和范围**：`cv2.calcHist([img], [0], None, [256], [0, 256])` 四个参数缺一不可，通道是 `[0]` 不是 `0`，范围是 `[0, 256]` 不是 `[0, 255]`。
> - **对彩色图直接 `calcHist`**：会默认只统计第 0 个通道（B 通道），不是亮度直方图。解决：先转灰度，或明确指定要统计的通道列表。

In [ ]:
# ============ 4.6 实例：灰度直方图计算与绘制（硬币图 + 指纹图 + grayLena） ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

# 读取三张灰度图
img_coin = cv2.imread('P&V/coin.jpg', cv2.IMREAD_GRAYSCALE)                         # 硬币图
img_fp = cv2.imread('P&V/fingerprint.png', cv2.IMREAD_GRAYSCALE)                    # 指纹图
img_lena = cv2.imread('P&V/grayLena.bmp', cv2.IMREAD_GRAYSCALE)                     # grayLena 图
assert img_coin is not None, '图像读取失败，请检查路径: P&V/coin.jpg'               # 断言
assert img_fp is not None, '图像读取失败，请检查路径: P&V/fingerprint.png'          # 断言
assert img_lena is not None, '图像读取失败，请检查路径: P&V/grayLena.bmp'           # 断言

# 计算直方图
hist_coin = cv2.calcHist([img_coin], [0], None, [256], [0, 256])                    # 硬币图直方图
hist_fp = cv2.calcHist([img_fp], [0], None, [256], [0, 256])                        # 指纹图直方图
hist_lena = cv2.calcHist([img_lena], [0], None, [256], [0, 256])                    # grayLena 图直方图

# 显示
plt.figure(figsize=(14, 15))                                                        # 画布

# ---------- 硬币图 ----------
ax1 = plt.subplot(3, 2, 1)                                                          # 硬币图
ax1.imshow(img_coin, cmap='gray')                                                   # 显示
ax1.set_title('Coin (Original)')                                                    # 标题
ax1.set_xticks([])                                                                  # 去掉刻度
ax1.set_yticks([])                                                                  # 去掉刻度
for spine in ax1.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 2)                                                                # 硬币图直方图
plt.plot(hist_coin)                                                                 # 绘制
plt.title('Histogram of Coin')                                                      # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ---------- 指纹图 ----------
ax3 = plt.subplot(3, 2, 3)                                                          # 指纹图
ax3.imshow(img_fp, cmap='gray')                                                     # 显示
ax3.set_title('Fingerprint (Original)')                                             # 标题
ax3.set_xticks([])                                                                  # 去掉刻度
ax3.set_yticks([])                                                                  # 去掉刻度
for spine in ax3.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 4)                                                                # 指纹图直方图
plt.plot(hist_fp)                                                                   # 绘制
plt.title('Histogram of Fingerprint')                                               # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ---------- grayLena 图 ----------
ax5 = plt.subplot(3, 2, 5)                                                          # grayLena 图
ax5.imshow(img_lena, cmap='gray')                                                   # 显示
ax5.set_title('GrayLena (Original)')                                                # 标题
ax5.set_xticks([])                                                                  # 去掉刻度
ax5.set_yticks([])                                                                  # 去掉刻度
for spine in ax5.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 6)                                                                # grayLena 图直方图
plt.plot(hist_lena)                                                                 # 绘制
plt.title('Histogram of GrayLena')                                                  # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：三张图直方图形态不同，反映各自灰度分布特征')                            # 打印结论

这三张直方图分别反映了三种完全不同类型的灰度分布，能直接看出图像的内容特征。

**1. Histogram of Coin（硬币图）**

- **最左侧（灰度 20~30）有一个极高的尖峰**，像素数超过 50 万，说明图像中绝大多数像素都是暗背景；
- **中间灰度区域几乎为空**，说明背景和硬币之间反差大，过渡像素少；
- **灰度 150~180 附近有一个小峰**，对应硬币表面金属反光区域，像素数远少于背景；
- **灰度 200 以上几乎为零**，没有接近纯白的高光。

**说明**：这是典型的“暗背景 + 亮主体”分布，背景占绝对多数。适合用**灰度级分层**或**阈值分割**把硬币从背景中分离出来。

**2. Histogram of Fingerprint（指纹图）**

- **灰度 20~30 附近有一个小峰**，对应指纹的黑色脊线，像素数约 5000；
- **中间灰度几乎为空**，说明脊线和白色谷线之间几乎没有过渡灰度，对比度极高；
- **灰度 250 附近有一个极高的尖峰**，像素数超过 45000，对应指纹的白色背景；
- **整体呈“两端高、中间空”的双峰分布**。

**说明**：这是典型的高对比度二值化图像分布，白色背景占多数，黑色脊线占少数。适合直接做**二值化**或**形态学处理**提取脊线。

**3. Histogram of GrayLena（Lena 灰度图）**

- **分布范围宽**，从灰度 50 一直延伸到 250；
- **多个峰和谷交替出现**，大致在 70、110、150、170、200 附近都有峰；
- **峰值高度相对接近**，没有像前两张那样出现压倒性的尖峰；
- **低灰度（0~50）和高灰度（250 以上）几乎没有像素**。

**说明**：这是典型的自然图像灰度分布——连续、宽广、多峰，反映 Lena 图中既有暗部（头发、阴影）、中间调（皮肤、帽子）、也有亮部（背景、帽子高光），灰度层次丰富。这类图像适合用**直方图均衡化**或 **CLAHE** 来进一步增强对比度。

**三张图对比总结**

| 图像 | 直方图形态 | 主要灰度 | 说明 |
|------|-----------|----------|------|
| Coin | 左尖峰 + 右小峰 | 暗背景为主 | 暗背景 + 亮主体 |
| Fingerprint | 两端双峰 | 白背景为主 | 白背景 + 黑线条，高对比度 |
| GrayLena | 宽范围多峰 | 中间灰度为主 | 自然图像，灰度层次丰富 |

**一句话总结**：Coin 是“暗背景 + 亮主体”，Fingerprint 是“白背景 + 黑线”的高对比度图，GrayLena 是灰度层次丰富的自然图像。直方图形态直接反映了图像的内容和对比度特征，也决定了后续该用哪种增强或分割方法。

---
## 4.7 直方图均衡化 (Histogram Equalization)

直方图均衡化自动调整对比度，使直方图分布更均匀。

**OpenCV 函数**：

```python
img_eq = cv2.equalizeHist(img_gray)
```

**应用**：医学影像增强、车牌识别、人脸检测。

> **常见坑**：
> - **直方图均衡化后噪声放大**：全局均衡化会把窄灰度区间强行拉伸到 `[0,255]`，该区域内的噪声也被一起放大。解决：用 CLAHE 限制对比度（详见 4.9）。
> - **对彩色图直接 `equalizeHist`**：`cv2.equalizeHist` 只接受单通道图，传三通道会报错。解决：先转灰度，或转 YCrCb 只对 Y 通道处理（详见 4.10）。
> - **均衡化后图像偏色**：如果对 B/G/R 三通道分别均衡化再合并，三个通道映射曲线不同，色调被破坏。解决：转 YCrCb 或 Lab，只对亮度通道均衡化。

In [ ]:
# ============ 4.7 实例：直方图均衡化 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/bone.jpg', cv2.IMREAD_GRAYSCALE)                         # 读取灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/bone.jpg'               # 断言图像读取成功

# 直方图均衡化
img_eq = cv2.equalizeHist(img_gray)                                                 # 均衡化

# 计算直方图
hist_orig = cv2.calcHist([img_gray], [0], None, [256], [0, 256])                    # 原图直方图
hist_eq = cv2.calcHist([img_eq], [0], None, [256], [0, 256])                        # 均衡化直方图

# 显示：上下两行，左边原图/均衡化图，右边对应直方图
plt.figure(figsize=(14, 10))                                                        # 画布

# ---------- 第一行：原图 + 原图直方图 ----------
ax1 = plt.subplot(2, 2, 1)                                                          # 原图
ax1.imshow(img_gray, cmap='gray')                                                   # 显示
ax1.set_title('Original')                                                           # 标题
ax1.set_xticks([])                                                                  # 去掉刻度
ax1.set_yticks([])                                                                  # 去掉刻度
for spine in ax1.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(2, 2, 2)                                                                # 原图直方图
plt.plot(hist_orig)                                                                 # 绘制
plt.title('Histogram of Original')                                                  # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ---------- 第二行：均衡化图 + 均衡化直方图 ----------
ax3 = plt.subplot(2, 2, 3)                                                          # 均衡化后
ax3.imshow(img_eq, cmap='gray')                                                     # 显示
ax3.set_title('Equalized')                                                          # 标题
ax3.set_xticks([])                                                                  # 去掉刻度
ax3.set_yticks([])                                                                  # 去掉刻度
for spine in ax3.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(2, 2, 4)                                                                # 均衡化直方图
plt.plot(hist_eq)                                                                   # 绘制
plt.title('Histogram of Equalized')                                                 # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：直方图均衡化使灰度分布更均匀，增强对比度')                              # 打印结论

---
## 4.8 直方图匹配 (Histogram Matching)

直方图匹配（也叫直方图规定化）将一张图的直方图匹配到另一张图的直方图。

**实现步骤**：

1. 计算源图和目标图的累积分布函数（CDF）；
2. 构建映射 LUT；
3. 应用 LUT。

**OpenCV 实现**：

```python
src_hist = cv2.calcHist([img_src], [0], None, [256], [0, 256]).ravel()
dst_hist = cv2.calcHist([img_dst], [0], None, [256], [0, 256]).ravel()
src_cdf = np.cumsum(src_hist) / np.sum(src_hist)
dst_cdf = np.cumsum(dst_hist) / np.sum(dst_hist)
lut = np.zeros(256, dtype=np.uint8)
for i in range(256):
    j = np.argmin(np.abs(dst_cdf - src_cdf[i]))
    lut[i] = j
img_matched = cv2.LUT(img_src, lut)
```

> **常见坑**：
> - **直方图匹配后图像偏色**：LUT 映射后像素可能超出 0~255，uint8 回绕导致颜色跳变。解决：对匹配结果 `np.clip` 到 0~255（如果是在浮点域计算），或确保 LUT 本身取值在 0~255。
> - **匹配后直方图和目标不一样**：匹配是单调映射，不能重排像素，只能让 CDF 接近。解决：接受尖峰/台阶，或换灰度层次更丰富的源图。
> - **`np.cumsum` 后忘了归一化**：不除以 `np.sum(...)`，CDF 不是 `[0,1]`，`np.argmin` 比较时量纲不一致。解决：`src_cdf = np.cumsum(src_hist) / np.sum(src_hist)`。

In [ ]:
# ============ 4.8 实例：直方图匹配 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_src = cv2.imread('P&V/bone.jpg', cv2.IMREAD_GRAYSCALE)                          # 源图
img_dst = cv2.imread('P&V/fingerprint.png', cv2.IMREAD_GRAYSCALE)                   # 目标图（指纹）
assert img_src is not None and img_dst is not None, '图像读取失败'                   # 断言

# 计算直方图
src_hist = cv2.calcHist([img_src], [0], None, [256], [0, 256]).ravel()              # 源图直方图
dst_hist = cv2.calcHist([img_dst], [0], None, [256], [0, 256]).ravel()              # 目标图直方图

# 计算 CDF
src_cdf = np.cumsum(src_hist) / np.sum(src_hist)                                    # 源图 CDF
dst_cdf = np.cumsum(dst_hist) / np.sum(dst_hist)                                    # 目标图 CDF

# 构建映射 LUT
lut = np.zeros(256, dtype=np.uint8)                                                 # 初始化 LUT
for i in range(256):                                                                # 遍历每个灰度级
    j = np.argmin(np.abs(dst_cdf - src_cdf[i]))                                     # 找最接近的 CDF
    lut[i] = j                                                                      # 记录映射

# 应用 LUT
img_matched = cv2.LUT(img_src, lut)                                                 # 直方图匹配

# 计算匹配后图像的直方图
matched_hist = cv2.calcHist([img_matched], [0], None, [256], [0, 256]).ravel()      # 匹配后直方图

# 显示：三行两列，左边图像，右边对应直方图
plt.figure(figsize=(14, 15))                                                        # 画布

# ---------- 第一行：源图 + 源图直方图 ----------
ax1 = plt.subplot(3, 2, 1)                                                          # 源图
ax1.imshow(img_src, cmap='gray')                                                    # 显示
ax1.set_title('Source (Bone)')                                                      # 标题
ax1.set_xticks([])                                                                  # 去掉刻度
ax1.set_yticks([])                                                                  # 去掉刻度
for spine in ax1.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 2)                                                                # 源图直方图
plt.plot(src_hist)                                                                  # 绘制
plt.title('Histogram of Source (Bone)')                                             # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ---------- 第二行：目标图 + 目标图直方图 ----------
ax3 = plt.subplot(3, 2, 3)                                                          # 目标图（指纹）
ax3.imshow(img_dst, cmap='gray')                                                    # 显示
ax3.set_title('Target (Fingerprint)')                                               # 标题
ax3.set_xticks([])                                                                  # 去掉刻度
ax3.set_yticks([])                                                                  # 去掉刻度
for spine in ax3.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 4)                                                                # 目标图直方图
plt.plot(dst_hist)                                                                  # 绘制
plt.title('Histogram of Target (Fingerprint)')                                      # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ---------- 第三行：匹配后 + 匹配后直方图 ----------
ax5 = plt.subplot(3, 2, 5)                                                          # 匹配后
ax5.imshow(img_matched, cmap='gray')                                                # 显示
ax5.set_title('Matched (Bone -> Fingerprint)')                                      # 标题
ax5.set_xticks([])                                                                  # 去掉刻度
ax5.set_yticks([])                                                                  # 去掉刻度
for spine in ax5.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 6)                                                                # 匹配后直方图
plt.plot(matched_hist)                                                              # 绘制
plt.title('Histogram of Matched')                                                   # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：直方图匹配使源图的灰度分布接近目标图（指纹）')                          # 打印结论

**直方图匹配的目标是“让匹配后的直方图形状接近目标直方图”，不是“完全一模一样”**。所以看起来“没有变得一样”，这是正常的。

**为什么不能完全一样**

直方图匹配做的是**单调映射**：把源图的每个灰度值 `i` 映射到目标 CDF 上最接近的灰度值 `j`。它有两个固有限制：

1. **映射是单调的，不能重排像素**。源图里灰度相同的像素，匹配后仍然是同一个灰度值；源图里没有的灰度，匹配后也不会凭空出现。所以匹配后的直方图，只能“重新分布”源图已有的像素，不能像目标图那样拥有完全相同的像素数分布。

2. **源图灰度级数量有限**。Bone 图的灰度主要集中在低值区，高值区像素极少。匹配后，大量低灰度像素被映射到同一个目标灰度上，就形成了你截图里那根**在灰度 128 附近的尖峰**——大量像素挤在同一个灰度值上。而目标指纹图的分布是“低值小峰 + 高值大峰”，匹配后只能近似，不可能完全复制。

**你截图里看到了什么**

- **Source（Bone）直方图**：像素集中在低灰度区，高灰度区几乎为空；
- **Target（Fingerprint）直方图**：低值小峰 + 高值大峰，中间几乎为空；
- **Matched 直方图**：确实出现了向目标靠拢的趋势——低值区仍有分布，高值区也有分布，中间出现了一根尖峰。但这根尖峰是“多个源灰度映射到同一个目标灰度”造成的，不是目标图本身的形状。

**所以结论是**

- **直方图匹配不能保证匹配后直方图和目标完全一样**，只能让它的**累积分布函数（CDF）**接近目标 CDF；
- 当源图灰度分布与目标图差异很大时，匹配结果会出现明显的“尖峰”或“台阶”，这是正常的；
- 如果希望匹配后直方图更接近目标，需要源图本身灰度层次足够丰富，或者改用其他方法（如直方图规定化的连续映射、或者对源图先做平滑）。

**一句话总结**：直方图匹配是“让 CDF 接近”，不是“让直方图相同”。源图和目标图差异越大，匹配后越不可能完全一样，出现尖峰是映射的必然结果。

---
## 4.9 限制对比度自适应直方图均衡化 (CLAHE)

### 4.9.1 从全局直方图均衡化的问题说起

4.7 节的 `cv2.equalizeHist` 是**全局**均衡化：它统计整幅图的直方图，然后用一条统一的映射曲线把灰度重新分布。这在整幅图亮度比较均匀时效果不错，但有两个明显问题：

- **噪声被放大**：如果图像某块区域灰度很集中（比如大片暗背景），均衡化会把这段窄区间强行拉伸到 `[0, 255]`，该区域里的微小噪声也被一起放大，出现颗粒感；
- **局部细节照顾不到**：全局只有一条映射曲线，暗部提亮的同时，亮部可能被压缩过度，局部对比度反而变差。

### 4.9.2 CLAHE 的两个关键词

CLAHE 的全称是 **Contrast Limited Adaptive Histogram Equalization**，中文叫“限制对比度自适应直方图均衡化”。它针对上面两个问题，分别做了两件事：

- **Adaptive（自适应）**：不再对整幅图做一次均衡化，而是把图像划分成若干个小块（tile），**每个小块各自做直方图均衡化**。这样每个局部区域都能根据自己的灰度分布独立增强，暗部、亮部的细节都能照顾到；
- **Contrast Limited（限制对比度）**：如果某个小块里灰度非常集中，直接均衡化会把噪声放大。CLAHE 在均衡化之前先**裁剪（clip）直方图**：把超过某个阈值的柱子削掉，把削掉的部分均匀分到其他灰度级上。这样映射曲线不会过陡，噪声不会被过度放大。

### 4.9.3 两个参数的含义

```python
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
img_clahe = clahe.apply(img_gray)
```

- **`clipLimit`**：对比度限制阈值。它决定直方图裁剪的力度。值越大，允许的对比度越强，噪声放大越明显；值越小，对比度增强越温和，噪声抑制越好。常用范围 `1.0~4.0`，默认 `2.0` 是比较稳妥的起点。
- **`tileGridSize`**：分块大小，格式是 `(列数, 行数)`。例如 `(8,8)` 表示把图像分成 8×8 共 64 个小块，每块各自均衡化。块越小，局部增强越强，但也越容易在块与块之间产生“拼缝”；块越大，越接近全局均衡化。常用 `(8,8)`。

### 4.9.4 块与块之间怎么过渡

如果每个小块独立均衡化，块与块边界处会出现明显的灰度跳变。CLAHE 用**双线性插值**解决：每个像素的最终取值，由它周围 4 个块中心的映射结果插值得到，所以块与块之间是平滑过渡的，不会出现明显拼缝。

### 4.9.5 什么时候用 CLAHE

- **医学影像**：X 光、CT、MRI 中局部对比度差异大，CLAHE 能同时看清暗部和亮部细节；
- **低对比度图像**：雾天、水下、夜间监控图像；
- **指纹、遥感图像**：局部纹理增强。

### 4.9.6 和全局均衡化的对比

| 方法 | 映射范围 | 噪声放大 | 局部细节 | 块间拼缝 |
|------|----------|----------|----------|----------|
| `equalizeHist` | 全局一条曲线 | 明显 | 照顾不到 | 无 |
| CLAHE | 每个小块各一条曲线 | 被 `clipLimit` 抑制 | 好 | 双线性插值消除 |

**一句话总结**：CLAHE = 分块均衡化 + 直方图裁剪 + 双线性插值。它比全局均衡化更“温柔”，能在增强局部对比度的同时抑制噪声放大，因此特别适合医学影像这类局部差异大的图像。

> **常见坑**：
> - **CLAHE 对象只能 apply 单通道图**：`clahe.apply(img_bgr)` 会报错。解决：先转灰度，或转 YCrCb/Lab 只对亮度通道 apply。
> - **`tileGridSize` 传成 `(8, 8, 8)` 或 `8`**：必须是 `(列数, 行数)` 的二元组。解决：写 `(8, 8)`。
> - **`clipLimit` 设得过大**：等于几乎不做对比度限制，退化成普通 AHE，噪声被放大。解决：常用 `1.0~4.0`，从 `2.0` 起调。

In [ ]:
# ============ 4.9 实例：CLAHE ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/bone.jpg', cv2.IMREAD_GRAYSCALE)                         # 读取灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/bone.jpg'               # 断言图像读取成功

# 创建 CLAHE 对象
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))                          # 创建
img_clahe = clahe.apply(img_gray)                                                   # 应用

# 显示
plt.figure(figsize=(12, 5))                                                         # 画布

plt.subplot(1, 2, 1)                                                                # 原图
plt.imshow(img_gray, cmap='gray')                                                   # 显示
plt.title('Original')                                                               # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 2, 2)                                                                # CLAHE
plt.imshow(img_clahe, cmap='gray')                                                  # 显示
plt.title('CLAHE')                                                                  # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：CLAHE 在增强对比度的同时抑制噪声放大')                                  # 打印结论

---
## 4.10 彩色图像均衡化策略 (Color Image Equalization)

### 4.10.1 为什么不能直接对 RGB 三通道分别均衡化

灰度图只有一个通道，直方图均衡化只有一条映射曲线。彩色图有 R、G、B 三个通道，如果**分别对三个通道做均衡化**：

```python
b, g, r = cv2.split(img_bgr)
b_eq = cv2.equalizeHist(b)
g_eq = cv2.equalizeHist(g)
r_eq = cv2.equalizeHist(r)
img_rgb_eq = cv2.merge((b_eq, g_eq, r_eq))
```

- `cv2.split` 把 BGR 拆成三个单通道；
- 每个通道各自算自己的直方图、各自算映射曲线；
- `cv2.merge` 再把三个处理后的通道按 B、G、R 的顺序叠回三通道图。

问题就在“各自算各自的”：三个通道的直方图分布不同，均衡化后各自的映射曲线也不同，原来 `B:G:R` 的比例被改变，**色调、饱和度都被破坏**，结果就是**偏色**——天空可能变青、肤色可能发紫、红色可能变橙，整体不自然。

> **注意**：`cv2.merge` 这一步不是数值相加，而是**通道堆叠**。每个像素 `(y, x)` 处，只是把均衡化后的 `b_eq[y, x]`、`g_eq[y, x]`、`r_eq[y, x]` 重新放回同一个像素的三个通道位置。偏色不是因为合并出错，而是因为三个通道的映射曲线不同，合并后通道比例已经被改变。

### 4.10.2 正确的策略：转到 YCrCb，只对 Y 均衡化

人眼对亮度敏感、对色度不敏感。彩色图像均衡化的核心策略是：**把亮度与色度分离，只增强亮度，色度保持不动**。YCrCb 就是这样一个空间：

- **Y**：亮度（Luma）；
- **Cr**：红色差（Chrominance Red）；
- **Cb**：蓝色差（Chrominance Blue）。

流程是：

```python
img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)   # BGR -> YCrCb
y, cr, cb = cv2.split(img_ycrcb)                          # 拆出 Y、Cr、Cb
y_eq = cv2.equalizeHist(y)                                # 只对 Y 均衡化
img_ycrcb_eq = cv2.merge((y_eq, cr, cb))                  # 合并，Cr、Cb 原样
img_color_eq = cv2.cvtColor(img_ycrcb_eq, cv2.COLOR_YCrCb2BGR)  # 转回 BGR
```

- `cvtColor`：把 BGR 转成 YCrCb，把亮度和色度分到不同通道；
- `split`：拆出 Y、Cr、Cb；
- `equalizeHist(y)`：只对 Y 做均衡化，Cr、Cb 完全不碰；
- `merge`：把处理后的 Y 和原来的 Cr、Cb 重新叠成三通道 YCrCb；
- `cvtColor`：再转回 BGR，方便显示和保存。

因为 Cr、Cb 没变，色调和饱和度就保持了；只有 Y 变了，所以亮度增强、颜色不变。同理，也可以在 Lab 空间只对 L 通道均衡化，效果类似，Lab 的 L 更接近人眼感知，YCrCb 计算更快，实际使用时任选一种。

### 4.10.3 两种做法的对比

| 方法 | 处理对象 | 亮度 | 颜色 | 视觉评价 |
|------|----------|------|------|----------|
| RGB 三通道分别均衡化 | B、G、R 各自独立 | 增强 | 失真、偏色 | 不自然，不推荐 |
| YCrCb 只对 Y 均衡化 | 只处理 Y，Cr、Cb 不动 | 增强 | 基本保持 | 自然，推荐 |

### 4.10.4 直方图为什么要统一用 Y 通道

为了在同一个口径下比较三种做法对“亮度分布”的影响，三行直方图都统计 **Y 通道**：

- 第一行：原图的 Y 通道直方图；
- 第二行：RGB 均衡化结果再转 YCrCb 后的 Y 通道直方图；
- 第三行：YCrCb 只对 Y 均衡化后的 Y 通道直方图。

这样能直观看出：原图 Y 分布集中在低灰度；RGB 均衡化后 Y 分布虽然也变了，但图像本身已偏色；YCrCb 均衡化后 Y 分布被摊开到 `[0, 255]`，图像颜色却保持。

> **注意**：第二行图像是对 B/G/R 三个通道分别均衡化的，但右边如果只画一条 Y 通道直方图，就看不出“三个通道各自被改成了什么样”。更准确的做法是第二行画 **B、G、R 三条直方图叠加**，第一、三行画 Y 通道，并在标题里注明口径不同。本节代码采用后一种做法。

### 4.10.5 用什么样的图演示

演示这个策略，必须用**颜色信息丰富**的图像，最好同时满足两个条件：

- **颜色丰富**：能看出 RGB 均衡化偏色、YCrCb 均衡化保色；
- **亮度有问题**：整体偏暗、逆光、阴影重、低对比度，能看出均衡化确实提亮了暗部。

典型图像包括：逆光人像、暗光彩色照片、曝光不足的风景照、雾天/阴天照。

灰度图或近似灰度的图（如硬币图）色度通道几乎没有变化，看不出颜色是否失真，不适合演示这个策略。`house.jpg` 颜色丰富但亮度正常，适合演示“保色”；`darkcolorful.png` 颜色丰富、整体偏暗，适合同时演示“提亮 + 保色”。

### 4.10.6 如果想更温和，可以把全局均衡化换成 CLAHE

```python
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
y_eq = clahe.apply(y)
```

其余步骤完全一样。CLAHE 分块均衡化 + 对比度限制 + 双线性插值，能在增强亮度的同时抑制噪声放大，比全局 `equalizeHist` 更适合彩色照片。

### 4.10.7 一句话总结

彩色图像均衡化的关键不是“对三个通道分别均衡化”，而是**把亮度与色度分离，只增强亮度、不动色度**。在 YCrCb（或 Lab）空间只对 Y（或 L）通道处理，就能在增强对比度的同时保持颜色自然；用一张**颜色丰富 + 亮度有问题**的图演示，效果最直观。

> **常见坑**：
> - **直接对 RGB 三通道分别均衡化导致偏色**：三个通道映射曲线不同，`B:G:R` 比例被破坏。解决：转到 YCrCb 或 Lab，只对 Y/L 通道均衡化。
> - **用近似灰度的图演示彩色均衡化，看不出颜色失真**：色度通道几乎没有变化。解决：换用颜色丰富的彩色图，最好同时亮度偏暗。
> - **第二行只画单 Y 直方图，看不出 RGB 均衡化对三个通道各自做了什么**：口径不统一，掩盖了 B/G/R 差异。解决：第二行画 B、G、R 三条直方图，第一、三行画 Y 通道，并注明口径。
> - **HSV 转 Lab 报错**：不能直接转。解决：先转回 BGR，再转 Lab。

In [ ]:
# ============ 4.10 实例：彩色图像均衡化 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_bgr = cv2.imread('P&V/darkcolorful.png')                                        # 读取彩色图
assert img_bgr is not None, '图像读取失败，请检查路径: P&V/darkcolorful.png'        # 断言图像读取成功

# ---------- 方法1：直接对 RGB 三通道分别均衡化（会偏色，作为反例） ----------
b, g, r = cv2.split(img_bgr)                                                        # 分离 B/G/R 通道
b_eq = cv2.equalizeHist(b)                                                          # 对 B 通道均衡化
g_eq = cv2.equalizeHist(g)                                                          # 对 G 通道均衡化
r_eq = cv2.equalizeHist(r)                                                          # 对 R 通道均衡化
img_rgb_eq = cv2.merge((b_eq, g_eq, r_eq))                                          # 合并通道

# ---------- 方法2：转 YCrCb，只对 Y 通道均衡化（保色策略） ----------
img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)                              # BGR -> YCrCb
y, cr, cb = cv2.split(img_ycrcb)                                                    # 分离通道
y_eq = cv2.equalizeHist(y)                                                          # 对 Y 通道均衡化
img_ycrcb_eq = cv2.merge((y_eq, cr, cb))                                            # 合并通道
img_color_eq = cv2.cvtColor(img_ycrcb_eq, cv2.COLOR_YCrCb2BGR)                      # YCrCb -> BGR

# ---------- 计算各自的 Y 通道直方图（统一用亮度通道对比） ----------
hist_orig = cv2.calcHist([img_ycrcb], [0], None, [256], [0, 256])                   # 原图 Y 通道直方图

img_rgb_eq_ycrcb = cv2.cvtColor(img_rgb_eq, cv2.COLOR_BGR2YCrCb)                    # RGB 均衡化结果转 YCrCb
hist_rgb_eq = cv2.calcHist([img_rgb_eq_ycrcb], [0], None, [256], [0, 256])          # RGB 均衡化后 Y 通道直方图

hist_y_eq = cv2.calcHist([img_ycrcb_eq], [0], None, [256], [0, 256])                # YCrCb Y 均衡化后直方图

# ---------- 显示：三行两列，左边图像，右边对应直方图 ----------
plt.figure(figsize=(14, 15))                                                        # 画布

# ========== 第一行：原图 + 原图 Y 通道直方图 ==========
ax1 = plt.subplot(3, 2, 1)                                                          # 原图
ax1.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))                                # 显示
ax1.set_title('Original (Dark Colorful)')                                           # 标题
ax1.set_xticks([])                                                                  # 去掉刻度
ax1.set_yticks([])                                                                  # 去掉刻度
for spine in ax1.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 2)                                                                # 原图 Y 通道直方图
plt.plot(hist_orig)                                                                 # 绘制
plt.title('Histogram of Original (Y channel)')                                      # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ========== 第二行：RGB 均衡化 + 其 Y 通道直方图 ==========
ax3 = plt.subplot(3, 2, 3)                                                          # RGB 均衡化
ax3.imshow(cv2.cvtColor(img_rgb_eq, cv2.COLOR_BGR2RGB))                             # 显示
ax3.set_title('RGB Channel-wise Equalization (Color Shift)')                        # 标题
ax3.set_xticks([])                                                                  # 去掉刻度
ax3.set_yticks([])                                                                  # 去掉刻度
for spine in ax3.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 4)                                                                # RGB 均衡化后 Y 通道直方图
plt.plot(hist_rgb_eq)                                                               # 绘制
plt.title('Histogram of RGB Equalized (Y channel)')                                 # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

# ========== 第三行：YCrCb Y 均衡化 + 其 Y 通道直方图 ==========
ax5 = plt.subplot(3, 2, 5)                                                          # YCrCb Y 均衡化
ax5.imshow(cv2.cvtColor(img_color_eq, cv2.COLOR_BGR2RGB))                           # 显示
ax5.set_title('YCrCb Y-channel Equalization')                                       # 标题
ax5.set_xticks([])                                                                  # 去掉刻度
ax5.set_yticks([])                                                                  # 去掉刻度
for spine in ax5.spines.values():                                                   # 给四条边加细线
    spine.set_visible(True)                                                         # 显示边框
    spine.set_linewidth(0.8)                                                        # 细线宽度
    spine.set_color('black')                                                        # 细线颜色

plt.subplot(3, 2, 6)                                                                # YCrCb Y 均衡化后直方图
plt.plot(hist_y_eq)                                                                 # 绘制
plt.title('Histogram of YCrCb Y Equalized')                                         # 标题
plt.xlabel('Gray Value')                                                            # X 轴
plt.ylabel('Pixel Count')                                                           # Y 轴
plt.xlim([0, 256])                                                                  # X 轴范围

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：直接对 RGB 通道均衡化会导致颜色失真；在 YCrCb 空间只对 Y 通道均衡化，能在增强亮度的同时保持颜色')  # 打印结论

---
## 4.11 2D 直方图与反向投影

### 4.11.1 2D 直方图 (H-S 平面)

计算色调 (Hue) 和饱和度 (Saturation) 的联合直方图。

**OpenCV 实现**：

```python
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
hist_2d = cv2.calcHist([hsv], [0, 1], None, [180, 256], [0, 180, 0, 256])
```

> **常见坑**：
> - **用近似灰度的图演示 2D 直方图，几乎为空**：色度通道几乎没有变化，H-S 分布集中。解决：换用颜色丰富、目标与背景颜色差异明显的彩色图。
> - **H 通道范围写成 `[0, 255]`**：OpenCV 中 HSV 的 H 范围是 `[0, 180]`（8 位图），不是 `[0, 255]`。解决：`calcHist` 的 range 用 `[0, 180, 0, 256]`。
> - **忘了 `np.log1p`**：2D 直方图像素数分布极不均，直接 `imshow` 大部分是深蓝。解决：显示时用 `np.log1p(hist_2d)` 压缩高值、抬高低值。

### 4.11.2 直方图反向投影 (Histogram Backprojection)

用于在图像中寻找与特定直方图分布相似的区域（如目标跟踪）。

**OpenCV 实现**：

```python
cv2.normalize(hist_2d, hist_2d, 0, 255, cv2.NORM_MINMAX)
back_proj = cv2.calcBackProject([hsv], [0, 1], hist_2d, [0, 180, 0, 256], 1)
```

> **常见坑**：
> - **反向投影结果大片黑色，看不出目标**：目标面积太小，或参考范围太窄。解决：换目标面积大的颜色（如红裙、天空、草地），或放宽参考范围。
> - **`calcBackProject` 的 range 和 `calcHist` 不一致**：一个写 `[0,180,0,256]`，另一个写 `[0,255,0,256]`，结果错位。解决：两处 range 必须一致。
> - **忘了先 `cv2.normalize`**：反向投影前通常要把 2D 直方图归一化到 `[0,255]`，否则结果动态范围不可控。解决：`cv2.normalize(hist_2d, hist_2d, 0, 255, cv2.NORM_MINMAX)`。

In [ ]:
# ============ 4.11 实例：2D 直方图与反向投影 ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_bgr = cv2.imread('P&V/portraitGrass.png')                                       # 读取彩色图
assert img_bgr is not None, '图像读取失败，请检查路径: P&V/portraitGrass.png'       # 断言图像读取成功

# 转换到 HSV 空间
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)                                      # BGR -> HSV

# 计算整幅图的 2D 直方图 (H: 0-180 -> 36 箱, S: 0-256 -> 32 箱)
hist_2d = cv2.calcHist([hsv], [0, 1], None, [36, 32], [0, 180, 0, 256])             # 计算
cv2.normalize(hist_2d, hist_2d, 0, 255, cv2.NORM_MINMAX)                            # 归一化
hist_2d_log = np.log1p(hist_2d)                                                     # 对数变换，抬高低值

# 反向投影：用红色区域的 H-S 分布，在整幅图中找红色物体
mask_red = (cv2.inRange(hsv, (0, 80, 60), (10, 255, 255)) |
            cv2.inRange(hsv, (170, 80, 60), (180, 255, 255)))                       # 红色范围
hist_red = cv2.calcHist([hsv], [0, 1], mask_red, [36, 32], [0, 180, 0, 256])        # 红色区域的 2D 直方图
cv2.normalize(hist_red, hist_red, 0, 255, cv2.NORM_MINMAX)                          # 归一化

back_proj = cv2.calcBackProject([hsv], [0, 1], hist_red, [0, 180, 0, 256], 1)       # 反向投影

# 反向投影归一化 + 对数变换，让中间层次也显示出来
back_proj_norm = cv2.normalize(back_proj, None, 0, 255, cv2.NORM_MINMAX)            # 归一化
back_proj_log = np.log1p(back_proj_norm)                                            # 对数变换
back_proj_log = cv2.normalize(back_proj_log, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)  # 再归一化并转 uint8

# 把反向投影叠加到原图上：高概率区域涂红
overlay = img_bgr.copy()                                                            # 拷贝原图
mask_highlight = back_proj_log > 80                                                 # 高概率阈值
overlay[mask_highlight] = (0, 0, 255)                                               # 高亮区域涂红

# 显示
plt.figure(figsize=(18, 5))                                                         # 画布

plt.subplot(1, 4, 1)                                                                # 原图
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))                                # 显示
plt.title('Original (Portrait on Grass)')                                           # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 4, 2)                                                                # 2D 直方图
plt.imshow(hist_2d_log, cmap='jet', aspect='auto', origin='lower')                  # 显示（对数 + 原点在左下）
plt.title('2D Histogram (H-S, log scale)')                                          # 标题
plt.xlabel('Saturation')                                                            # X 轴
plt.ylabel('Hue')                                                                   # Y 轴
plt.colorbar(label='log(1 + count)')                                                # 颜色条

plt.subplot(1, 4, 3)                                                                # 反向投影（对数显示）
plt.imshow(back_proj_log, cmap='gray')                                              # 显示
plt.title('Back Projection (Find Red)')                                             # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 4, 4)                                                                # 叠加结果
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))                                # 显示
plt.title('Overlay: Red Region Highlighted')                                        # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：2D 直方图描述 H-S 联合分布（对数显示）；反向投影用红色分布找出红色目标；叠加图把高概率区域涂红，直观展示定位结果')  # 打印结论

**2D 直方图的读法与对数显示**

**1. 2D 直方图在统计什么**

2D 直方图（H-S 平面）统计的是：**整幅图里“什么颜色”出现了多少**。

- **横轴 Saturation（饱和度）**：0 是灰色/白色，越大越鲜艳；
- **纵轴 Hue（色调）**：把 0~180 的色调分箱，不同数值对应不同颜色；
- **每个格子的值**：落在该 H-S 组合的像素数；
- **颜色条**：值越大颜色越暖（红/黄），值越小颜色越冷（蓝）。

所以一张 2D 直方图，本质上是把“颜色分布”画成了一张二维热力图。

**2. 怎么读一张 2D 直方图**

以 `portraitGrass.png` 为例，图中几个明显区域分别对应：

| 区域 | H 值 | S 值 | 对应内容 |
|------|------|------|----------|
| 左下角红黄块 | H≈0~2 | S≈0~5 | 暗部、阴影、深色头发 |
| 中间偏下亮带 | H≈1~3 | S≈15~30 | 红裙、红上衣、红唇 |
| 中间偏上黄绿块 | H≈5~10 | S≈15~30 | 肤色、金发 |
| 上方青色横带 | H≈15~20 | S 宽 | 绿色草地 |
| 顶部深蓝区 | H≈20~35 | S 低 | 少量暗色、边缘 |
| 极小黄点 | H≈0 | S≈0 | 白裙、白花、高光 |

读图要点：

- **色调决定“是什么颜色”**：红色目标集中在 H≈1~3，草地集中在 H≈15~20；
- **饱和度决定“有多鲜艳”**：低 S 是灰白，高 S 是鲜艳；
- **颜色条决定“有多少像素”**：红/黄表示像素多，蓝表示像素少；
- **峰的位置决定“目标颜色是否独特”**：红色峰越集中、离其他峰越远，反向投影越干净。

**3. 为什么用 `log(1+count)`**

2D 直方图的像素数分布**极度不均**：

- 少数格子（如草地、红色目标）像素数可能上万；
- 大多数格子只有几十、甚至 0。

如果直接按原始值显示：

- 上万的高值把颜色条拉到很大范围；
- 几十、几百的低值全被压成同一个深蓝；
- 结果整幅图大部分是深蓝，只有几个亮点，信息量很少。

`log(1+count)` 就是对像素数做一次**对数变换**：

- `count`：某个 H-S 格子里的像素数；
- `1 + count`：加 1 避免 `log(0)` 无意义；
- `log(...)`：压缩高值、抬高低值。

效果：

- 高值被压缩，低值被抬高，**数值范围变小**；
- 几十、几百的格子也能显示层次；
- 整幅图的 H-S 结构就看得清楚了。

**4. 是以 10 为底吗**

不是固定的。常见三种：

| 写法 | 底数 | 说明 |
|------|------|------|
| `np.log1p(count)` | 自然对数 e | NumPy 默认，代码里常用 |
| `np.log2(1+count)` | 2 | 信息论常用 |
| `np.log10(1+count)` | 10 | 工程常用，量级更直观 |

不同底数只差一个常数倍数，而直方图后面还要 `normalize` 到 `[0, 255]`，这个常数倍数会被归一化抵消。所以三种写法显示出来的**形状几乎一样**，区别只在数值刻度，不影响“压缩高值、抬高低值”的效果。

**5. 一句话总结**

2D 直方图统计“什么颜色出现了多少”，横轴是饱和度、纵轴是色调、颜色条是像素数；读图时看峰的位置判断目标颜色是否独特，看峰的集中程度判断反向投影是否干净。由于像素数分布极不均，显示时通常做 `log(1+count)` 对数变换，压缩高值、抬高低值，让中间层次也能显示出来；`np.log1p` 用的是自然对数 e，换成 `log2` 或 `log10` 显示形状几乎一样。

**寻找红色（Find Red）**

**1. 目标**

在图像中找出“红色”的像素或区域。红色在 HSV 空间里有明确的取值范围：

- H 接近 0（约 0~10）或接近 180（约 170~180）；
- S 较高，表示颜色鲜艳；
- V 较高，表示亮度足够。

只要像素的 H、S、V 落在这些范围内，就认为是红色。

**2. 最简单的方法：`inRange`**

```python
mask_red = (cv2.inRange(hsv, (0, 80, 60), (10, 255, 255)) |
            cv2.inRange(hsv, (170, 80, 60), (180, 255, 255)))
```

- 红色在 H 轴上分两段，所以要写两个 `inRange`，再按位或；
- 得到的是二值掩膜：红色像素为 255，其余为 0；
- 简单、直接，不需要 2D 直方图，也不需要反向投影。

**3. 更平滑的方法：2D 直方图 + 反向投影**

- 先用 `inRange`（或手动掩膜）圈出红色参考区域；
- 只对参考区域算 H-S 的 2D 直方图，得到“红色长什么样”；
- 用这份分布对整幅图做反向投影，得到“每个像素有多像红色”；
- 结果是灰度图：越白越像红色，越黑越不像；
- 好处是边缘更平滑，更接近“概率式筛选”，而不是一刀切。

**4. 两种方法的区别**

| 方法 | 输出 | 边缘 | 阈值 | 是否需要 2D 直方图 |
|------|------|------|------|---------------------|
| `inRange` | 二值掩膜 | 硬、一刀切 | 人工指定 H/S/V | 不需要 |
| 2D 直方图 + 反向投影 | 灰度概率图 | 平滑 | 从参考区域自动学出 | 需要 |

**5. 结果好坏取决于什么**

- **目标颜色是否独特**：红色和背景差异越大，结果越干净；
- **目标面积是否大**：面积大，反向投影后大面积亮，效果好；面积小，只亮几个像素，容易被忽略；
- **参考范围是否合适**：
  - 范围太窄，红色边缘、暗部会被漏掉；
  - 范围太宽，肤色、暗红阴影会被误包含，结果变“脏”；
- **显示方式**：灰度显示不显眼时，可以叠加到原图、局部放大、或用彩色显示。

**6. 常见问题**

- **红裙亮、嘴唇不亮**：嘴唇面积小、颜色偏暗、周围有肤色干扰，反向投影后只是几个亮点，容易被忽略；
- **肤色、金发也被点亮**：肤色、金发和红色在 H-S 平面上有重叠，参考范围太宽时会误包含；
- **红色区域边缘断裂**：`inRange` 的硬阈值会切掉边缘过渡像素，反向投影能缓解这个问题。

**7. 一句话总结**

寻找红色的核心是**按 HSV 范围筛选像素**：最简单的方式是 `inRange`，得到二值掩膜；更平滑的方式是先圈出红色参考区域、算 H-S 的 2D 直方图、再做反向投影，得到灰度概率图。结果好坏取决于目标颜色是否独特、面积是否大、参考范围是否合适，以及显示方式是否直观。

> **常见坑**：
> - **`inRange` 结果全黑**：阈值范围不对。解决：先查看目标空间的取值范围（HSV 的 H 是 0~180，S/V 是 0~255）。
> - **红色只写了一段范围**：红色在 H 轴上分两段（0~10 和 170~180），只写一段会漏掉一部分红色。解决：两个 `inRange` 按位或。
> - **HSV 转 Lab 报错**：不能直接转。解决：必须先转回 BGR，再转 Lab。

---
## 4.12 噪声模型与去噪基础

### 4.12.1 为什么要提出噪声模型

去噪不能凭空去做，必须先知道噪声是什么样子，才能选对方法。提出噪声模型，是为了回答三个问题：

1. **噪声是什么**：它是高斯、椒盐、泊松，还是乘性噪声？
2. **该用什么方法去**：不同噪声对应不同滤波器；
3. **去得怎么样**：PSNR、SSIM 等指标需要已知噪声模型才能评估。

没有噪声模型，去噪就是盲试；有了噪声模型，才能选对方法、解释现象、量化效果。

**不同噪声，要用不同方法去除**：

- **高斯噪声**：适合非局部均值、双边滤波、高斯滤波；
- **椒盐噪声**：适合中值滤波，用均值滤波反而会把黑白点抹开；
- **泊松噪声**：适合方差稳定变换后再去噪；
- **乘性噪声**：适合同态滤波，先取对数把乘性变加性。

如果不先建立噪声模型，就可能“用错药”：拿中值滤波去高斯噪声，效果一般；拿均值滤波去椒盐噪声，黑点会变成灰斑。

**噪声模型也是评估去噪效果的前提**：

- 要知道“去噪后好了多少”，必须先知道“原来噪声有多大”；
- PSNR、SSIM 这些指标，都是在已知“干净图 + 噪声模型”的条件下才有意义；
- 实验中常按噪声模型人为加噪（如高斯 σ=25、椒盐 p=0.05），再去噪，才能定量比较不同方法。

**噪声模型还帮助理解“为什么去噪会失败”**：

- 椒盐噪声用均值滤波，黑点被平均成灰斑，看起来“没去掉”；
- 高斯噪声用中值滤波，平滑不够，噪点仍在；
- 乘性噪声不做同态变换，直接线性滤波，信号强的地方噪声更强，去不干净。

### 4.12.2 噪声模型

- **高斯噪声**：概率密度函数服从正态分布，常见于传感器热噪声；
- **椒盐噪声**：随机出现的黑白像素点，常见于传输错误；
- **泊松噪声**：服从泊松分布，常见于低光照条件；
- **乘性噪声**：与图像信号相乘，常见于雷达和超声图像。

### 4.12.3 去噪方法

- **非局部均值 (Non-Local Means)**：`cv2.fastNlMeansDenoising`，利用图像中相似块的冗余信息去噪；
- **双边滤波 (Bilateral Filter)**：`cv2.bilateralFilter`，保边去噪。

**OpenCV 实现**：

```python
img_nlm = cv2.fastNlMeansDenoising(img_noisy, None, h=10, templateWindowSize=7, searchWindowSize=21)
img_bilateral = cv2.bilateralFilter(img_noisy, d=9, sigmaColor=75, sigmaSpace=75)
```

> **常见坑**：
> - **不知道噪声类型就乱选滤波器**：不同噪声对应不同最优滤波器。解决：先建立噪声模型（高斯 / 椒盐 / 泊松 / 乘性），再选方法：高斯噪声用 NLM / 双边滤波，椒盐噪声用中值滤波，乘性噪声用同态滤波。
> - **高斯噪声直接加到 uint8 图上导致负值被截断**：`np.random.normal` 产生负值，`astype(np.uint8)` 会截断为 0。解决：先转 `float32` 相加，再 `np.clip(0,255)`，最后转 `uint8`。
> - **去噪后图像模糊**：滤波核过大。解决：减小核大小或调整参数。
> - **NLM 去噪速度慢**：搜索窗口过大。解决：减小 `searchWindowSize`。
> - **用中值滤波去高斯噪声、用均值滤波去椒盐噪声**：用错药，效果差。解决：按噪声类型选滤波器。

In [ ]:
# ============ 4.12 实例：噪声模型与去噪 ============
# 本节完全独立：自己导入库、自己读取图像
# 注意：fingerprint.png 是干净指纹图，本节在代码中自行加噪

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/fingerprint.png', cv2.IMREAD_GRAYSCALE)                  # 读取干净的指纹灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/fingerprint.png'        # 断言图像读取成功

# 1. 添加高斯噪声 (σ=40，比 σ=25 更明显；先加浮点噪声，再截断回 uint8，避免负值被截成 0)
gaussian_noise = np.random.normal(0, 40, img_gray.shape)                            # 浮点高斯噪声
img_gaussian = np.clip(img_gray.astype(np.float32) + gaussian_noise, 0, 255).astype(np.uint8)  # 加噪并截断

# 2. 添加块状椒盐噪声 (每个噪点 block×block)
img_salt_pepper = img_gray.copy()                                                   # 拷贝
block = 5                                                                           # 每个噪点块大小 5×5
prob = 0.02                                                                         # 块数量比例，块越大 prob 越小

num_blocks = int(prob * (img_gray.size / (block * block)) * 0.5)                    # 盐块数量（约一半）

for _ in range(num_blocks):                                                         # 随机撒盐块
    y = np.random.randint(0, img_gray.shape[0] - block)                             # 随机 y 坐标
    x = np.random.randint(0, img_gray.shape[1] - block)                             # 随机 x 坐标
    img_salt_pepper[y:y+block, x:x+block] = 255                                     # 置为白色盐块

for _ in range(num_blocks):                                                         # 随机撒椒块
    y = np.random.randint(0, img_gray.shape[0] - block)                             # 随机 y 坐标
    x = np.random.randint(0, img_gray.shape[1] - block)                             # 随机 x 坐标
    img_salt_pepper[y:y+block, x:x+block] = 0                                       # 置为黑色椒块

# 3. 去噪：非局部均值 (针对高斯噪声)
img_nlm = cv2.fastNlMeansDenoising(img_gaussian, None, h=10, templateWindowSize=7, searchWindowSize=21)  # NLM

# 4. 去噪：双边滤波 (保边)
img_bilateral = cv2.bilateralFilter(img_gaussian, d=9, sigmaColor=75, sigmaSpace=75)  # 双边滤波

# 5. 去噪：中值滤波 (针对椒盐噪声，窗口要比 block 大)
img_median = cv2.medianBlur(img_salt_pepper, 7)                                     # 7×7 中值滤波

# 显示
plt.figure(figsize=(15, 10))                                                        # 画布

plt.subplot(2, 3, 1)                                                                # 原图
plt.imshow(img_gray, cmap='gray')                                                   # 显示
plt.title('Original')                                                               # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 3, 2)                                                                # 高斯噪声
plt.imshow(img_gaussian, cmap='gray')                                               # 显示
plt.title('Gaussian Noise (sigma=40)')                                              # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 3, 3)                                                                # 块状椒盐噪声
plt.imshow(img_salt_pepper, cmap='gray')                                            # 显示
plt.title(f'Salt & Pepper Noise (block={block})')                                   # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 3, 4)                                                                # NLM 去噪
plt.imshow(img_nlm, cmap='gray')                                                    # 显示
plt.title('NLM Denoising')                                                          # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 3, 5)                                                                # 双边滤波
plt.imshow(img_bilateral, cmap='gray')                                              # 显示
plt.title('Bilateral Filtering')                                                    # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(2, 3, 6)                                                                # 中值滤波
plt.imshow(img_median, cmap='gray')                                                 # 显示
plt.title('Median Filtering (7x7)')                                                 # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print(f'结论：块状椒盐噪声（block={block}）噪点明显变大；中值滤波 7×7 能有效去除块状椒盐噪声')  # 打印结论

**关于第二行去噪结果的说明**：NLM 和双边滤波都是从高斯噪声图（sigma=40）降噪得到的；块状椒盐噪声图没有参与这两个滤波器的输入，因为椒盐噪声要用中值滤波，不是 NLM / 双边滤波。

---
## 4.13 去噪效果评估：PSNR 与 SSIM

### 4.13.1 为什么需要客观指标

去噪后，肉眼可以看“干不干净”，但：

- 不同方法谁更好？肉眼难分；
- 参数调大调小，到底改善了多少？肉眼只能感觉“差不多”；
- 论文、工程里需要**可比较的数字**，不能只靠“看起来还行”。

所以需要客观指标，把“去噪质量”变成一个数。PSNR 和 SSIM 就是最常用的两个。

### 4.13.2 PSNR（峰值信噪比）

**公式**：

$$PSNR = 10 \cdot \log_{10}\left(\frac{MAX^2}{MSE}\right)$$

- `MAX`：像素最大值，8 位图是 255；
- `MSE`：去噪图和干净图之间的均方误差：

$$MSE = \frac{1}{MN}\sum_{i=1}^{M}\sum_{j=1}^{N}(I_{clean}(i,j) - I_{denoised}(i,j))^2$$

**含义**：

- MSE 越小，说明去噪图和干净图越接近；
- PSNR 越大，说明失真越小，去噪效果越好；
- 单位是 **dB**（分贝）。

**典型数值**：

| 情况 | PSNR |
|------|------|
| 含噪图 vs 干净图 | 20~25 dB |
| 一般去噪结果 | 28~32 dB |
| 好的去噪结果 | 32~38 dB |
| 几乎无损 | 40 dB 以上 |

**PSNR 的问题**：

- 只看像素误差，**不关心结构**；
- 一张整体模糊的图，PSNR 可能不低，但结构已经坏了；
- 所以还需要 SSIM。

> **关于 PSNR 在实践中的适用边界**：PSNR 需要“干净参考图”才能计算，真实场景里往往没有干净图。详见附录 A。

### 4.13.3 SSIM（结构相似性）

**含义**：

- 不只看像素差，还看**亮度、对比度、结构**三个方面的相似度；
- 取值 `0~1`，越接近 1 越相似；
- 1 表示完全一样。

**公式（简化）**：

$$SSIM = \frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$$

- `μ`：均值（亮度）；
- `σ`：标准差（对比度）；
- `σ_xy`：协方差（结构）；
- `c₁`、`c₂`：防止分母为 0 的小常数。

**典型数值**：

| 情况 | SSIM |
|------|------|
| 含噪图 vs 干净图 | 0.4~0.6 |
| 一般去噪结果 | 0.7~0.85 |
| 好的去噪结果 | 0.85~0.95 |
| 几乎无损 | 0.95 以上 |

### 4.13.4 两个指标怎么配合看

| 情况 | PSNR | SSIM | 说明 |
|------|------|------|------|
| 去噪后更干净 | 升高 | 升高 | 真正改善 |
| 去噪后整体模糊 | 可能升高 | 下降 | 像素误差小，但结构被破坏 |
| 去噪后结构保留好 | 升高 | 升高 | 理想结果 |
| 去噪过度 | 下降 | 下降 | 细节被抹掉，反而不如不去 |

**所以**：

- PSNR 看“像素误差”；
- SSIM 看“结构保持”；
- 两个都高，才是真正的好去噪。

### 4.13.5 代码里怎么用

```python
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

psnr_val = psnr(img_clean, img_denoised)      # 越大越好
ssim_val = ssim(img_clean, img_denoised)      # 越接近 1 越好

print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')
```

**注意**：

- 第一个参数是**干净图**，第二个是**去噪图**；
- 顺序不能反，虽然 PSNR 对称，但 SSIM 在某些实现里对顺序敏感；
- 如果是彩色图，`ssim` 需要加 `channel_axis=2` 或 `multichannel=True`。

### 4.13.6 本节演示逻辑

代码里做的是：

1. 读干净指纹图 `img_gray`；
2. 加高斯噪声得到 `img_noisy`；
3. 用 NLM 去噪得到 `img_denoised`；
4. 用 `psnr(img_gray, img_denoised)` 和 `ssim(img_gray, img_denoised)` 算指标；
5. 打印 PSNR、SSIM，并和含噪图对比。

**典型输出**：

```
NLM 去噪后的 PSNR: 30.5 dB
NLM 去噪后的 SSIM: 0.85
```

- 含噪图的 PSNR 可能只有 20 dB 左右，SSIM 0.5 左右；
- 去噪后 PSNR 升到 30 dB、SSIM 升到 0.85，说明去噪确实改善了图像质量。

### 4.13.7 一句话总结

PSNR 和 SSIM 是量化去噪效果的两个指标：

- **PSNR**：看像素误差，越大越好，单位 dB；
- **SSIM**：看亮度、对比度、结构的相似度，越接近 1 越好；
- 两者结合，才能判断“去噪后是真正变好，还是只是变糊”。
- PSNR 需要干净参考图，真实场景没有参考时不能直接算，详见附录 A。

> **常见坑**：
> - **对干净图直接做去噪评估没有参考基准**：没有“带噪图 vs 干净图”的对照。解决：先自行加噪，再评估 PSNR/SSIM。
> - **真实场景没有干净图，不能直接算 PSNR**：PSNR 需要干净参考图；单张含噪照片没有参考。解决：用无参考指标、人眼主观评价、任务驱动评估；详见附录 A。
> - **PSNR 高但 SSIM 低**：PSNR 只看像素误差，不关心结构；去噪后整体模糊时 PSNR 可能不低，但结构已被破坏。解决：两个指标一起看；SSIM 下降说明结构没保住，需要调整滤波器参数或换方法。
> - **`ssim` 对彩色图报错**：需要加 `channel_axis=2` 或 `multichannel=True`。解决：按当前 scikit-image 版本传入正确参数。
> - **`psnr` / `ssim` 参数顺序反了**：虽然 PSNR 对称，但 SSIM 在某些实现里对顺序敏感。解决：第一个参数是干净图，第二个是去噪图。

In [ ]:
# ============ 4.13 实例：去噪效果评估 ============
# 本节完全独立：自己导入库、自己读取图像
# 注意：fingerprint.png 是干净指纹图，本节在代码中自行加噪后再评估

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib
from skimage.metrics import peak_signal_noise_ratio as psnr                         # PSNR
from skimage.metrics import structural_similarity as ssim                           # SSIM

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_gray = cv2.imread('P&V/fingerprint.png', cv2.IMREAD_GRAYSCALE)                  # 读取干净的指纹灰度图
assert img_gray is not None, '图像读取失败，请检查路径: P&V/fingerprint.png'        # 断言图像读取成功

# 添加高斯噪声 (先加浮点噪声，再截断回 uint8)
gaussian_noise = np.random.normal(0, 25, img_gray.shape)                            # 浮点高斯噪声
img_noisy = np.clip(img_gray.astype(np.float32) + gaussian_noise, 0, 255).astype(np.uint8)  # 加噪并截断

# NLM 去噪
img_denoised = cv2.fastNlMeansDenoising(img_noisy, None, h=10, templateWindowSize=7, searchWindowSize=21)  # 去噪

# 计算 PSNR 和 SSIM
psnr_val = psnr(img_gray, img_denoised)                                             # PSNR
ssim_val = ssim(img_gray, img_denoised)                                             # SSIM

print(f'NLM 去噪后的 PSNR: {psnr_val:.2f} dB')                                       # 打印 PSNR
print(f'NLM 去噪后的 SSIM: {ssim_val:.4f}')                                          # 打印 SSIM

# 显示
plt.figure(figsize=(15, 5))                                                         # 画布

plt.subplot(1, 3, 1)                                                                # 原图
plt.imshow(img_gray, cmap='gray')                                                   # 显示
plt.title('Original')                                                               # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 3, 2)                                                                # 含噪图
plt.imshow(img_noisy, cmap='gray')                                                  # 显示
plt.title('Noisy')                                                                  # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 3, 3)                                                                # 去噪后
plt.imshow(img_denoised, cmap='gray')                                               # 显示
plt.title(f'Denoised (PSNR={psnr_val:.2f} dB, SSIM={ssim_val:.4f})')                # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：PSNR 和 SSIM 量化去噪效果')                                             # 打印结论

---
## 4.14 图像修复 (Inpainting)

**1. 图像修复在做什么**

用周围已知的像素，去推断并补上图像中缺失或损坏的区域（划痕、污渍、文字、水印等）。

**2. 它怎么知道“哪里是错的”**

**不是算法自动判断，而是人给出一个掩膜（mask）**：

- 掩膜里为 255 的区域 = **待修复区域**；
- 掩膜里为 0 的区域 = **保留区域**；
- 掩膜是**人工指定的**，不是算法自动检测的。

所以“哪里坏了”是人的判断，算法只负责“怎么补”。

**3. 它凭什么去“补”**

知道“哪里坏了”之后，算法用**周围已知像素**去推断缺失区域应该是什么，依据有三类：

| 依据 | 含义 | 适用场景 |
|------|------|----------|
| 局部连续性 | 相邻像素灰度/颜色变化连续 | 小划痕、小污点、细线 |
| 边缘与结构延续 | 边缘、线条连续穿过缺失区域 | 边缘穿过缺失区域 |
| 纹理与统计相似性 | 周围有相似纹理，可“贴”过来 | 大面积缺失、纹理复杂 |

**4. 三种依据对应的典型方法**

- **局部连续性 + 边缘延续**：`cv2.INPAINT_TELEA`、`cv2.INPAINT_NS`；
- **纹理相似性**：PatchMatch、Criminisi 等基于块的修复方法。

**5. 为什么修复结果不总是完美**

因为修复是**基于假设的推断**，不是“真正还原”：

- 缺失区域小、周围结构简单 → 修复效果好；
- 缺失区域大、周围纹理复杂 → 结果可能不自然；
- 缺失区域正好是关键结构（如人脸的眼睛、文字的笔画）→ 算法只能“猜”，猜得不一定对；
- 修复结果**不是唯一正确答案**，只是“看起来合理”的一种补法。

**6. 和“图像复原”的区别**

| 方法 | 依据 | 典型场景 |
|------|------|----------|
| 图像修复 Inpainting | 周围像素的连续性、结构、纹理 | 划痕、污渍、文字、水印 |
| 图像复原 Restoration | 已知模糊核或噪声模型 | 运动模糊、失焦模糊、噪声退化 |

两者都叫“恢复”，但**依据不同**：修复靠“周围像素”，复原靠“退化模型”。

**7. 一句话总结**

图像修复**不是自动判断哪里错了**，而是**由人给出掩膜**，告诉算法“这些像素是坏的”；算法**依据周围已知像素的局部连续性、边缘结构、纹理相似性**去推断缺失区域应该是什么。修复结果**不是唯一正确答案**，只是“看起来合理”的一种补法；缺失区域越小、周围结构越简单，修复效果越好。

使用 `cv2.inpaint` 修复图像中的划痕或缺失区域。

- `INPAINT_TELEA`：基于快速行进方法；
- `INPAINT_NS`：基于纳维-斯托克斯方程。

**OpenCV 实现**：

```python
img_inpaint = cv2.inpaint(img_gray, mask, 3, cv2.INPAINT_TELEA)
```

> **常见坑**：
> - **图像修复后边缘不自然**：掩膜过大。解决：减小掩膜范围。
> - **掩膜数据类型不对**：`cv2.inpaint` 要求掩膜是 `uint8` 单通道，且非零像素表示待修复区域。解决：`mask = np.zeros((h, w), dtype=np.uint8)`，待修复区域置 255。
> - **掩膜和图像尺寸不一致**：`cv2.inpaint` 要求掩膜与图像同尺寸。解决：用 `img.shape[:2]` 创建掩膜。
> - **修复半径过小/过大**：半径过小，修复不干净；半径过大，边缘被过度平滑。解决：常用 `3`，根据缺失区域大小调整。

In [ ]:
# ============ 4.15 实例：图像修复（人为加污损 + 去除） ============
# 本节完全独立：自己导入库、自己读取图像

import numpy as np                                                                  # 导入 NumPy
import cv2                                                                          # 导入 OpenCV
import matplotlib.pyplot as plt                                                     # 导入 Matplotlib

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False                                          # 正常显示负号

img_bgr = cv2.imread('P&V/man.jpg')                                                 # 读取彩色图
assert img_bgr is not None, '图像读取失败，请检查路径: P&V/man.jpg'                  # 断言图像读取成功

h, w = img_bgr.shape[:2]                                                            # 图像尺寸

# ---------- 1. 人为制造污损 ----------
img_damaged = img_bgr.copy()                                                        # 拷贝原图
mask = np.zeros((h, w), dtype=np.uint8)                                             # 初始化掩膜

# 污损 1：斜划过左脸的划痕
cv2.line(img_damaged, (int(w*0.30), int(h*0.35)), (int(w*0.42), int(h*0.60)), (255, 255, 255), 5)  # 白色划痕
cv2.line(mask, (int(w*0.30), int(h*0.35)), (int(w*0.42), int(h*0.60)), 255, 5)      # 掩膜记录

# 污损 2：右脸的黑色污点
cv2.circle(img_damaged, (int(w*0.65), int(h*0.45)), 12, (0, 0, 0), -1)              # 黑色圆点
cv2.circle(mask, (int(w*0.65), int(h*0.45)), 12, 255, -1)                           # 掩膜记录

# 污损 3：额头的白色污点
cv2.circle(img_damaged, (int(w*0.45), int(h*0.18)), 10, (255, 255, 255), -1)        # 白色圆点
cv2.circle(mask, (int(w*0.45), int(h*0.18)), 10, 255, -1)                           # 掩膜记录

# 污损 4：领口的短划痕
cv2.line(img_damaged, (int(w*0.40), int(h*0.85)), (int(w*0.60), int(h*0.88)), (255, 255, 255), 4)  # 白色划痕
cv2.line(mask, (int(w*0.40), int(h*0.85)), (int(w*0.60), int(h*0.88)), 255, 4)      # 掩膜记录

# ---------- 2. 应用修复 ----------
img_inpaint_telea = cv2.inpaint(img_damaged, mask, 3, cv2.INPAINT_TELEA)            # TELEA 修复
img_inpaint_ns = cv2.inpaint(img_damaged, mask, 3, cv2.INPAINT_NS)                   # NS 修复

# ---------- 3. 显示 ----------
plt.figure(figsize=(16, 5))                                                         # 画布

plt.subplot(1, 4, 1)                                                                # 原图
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))                                # 显示
plt.title('Original (man.jpg)')                                                     # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 4, 2)                                                                # 污损图
plt.imshow(cv2.cvtColor(img_damaged, cv2.COLOR_BGR2RGB))                            # 显示
plt.title('Damaged (Scratch + Dots)')                                               # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 4, 3)                                                                # TELEA 修复
plt.imshow(cv2.cvtColor(img_inpaint_telea, cv2.COLOR_BGR2RGB))                      # 显示
plt.title('TELEA Inpainting')                                                       # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.subplot(1, 4, 4)                                                                # NS 修复
plt.imshow(cv2.cvtColor(img_inpaint_ns, cv2.COLOR_BGR2RGB))                         # 显示
plt.title('NS Inpainting')                                                          # 标题
plt.axis('off')                                                                     # 关闭坐标轴

plt.tight_layout()                                                                  # 调整间距
plt.show()                                                                          # 渲染

print('结论：人为污损后，inpaint 可有效去除划痕和污点；TELEA 边缘更平滑，NS 更保结构')  # 打印结论

---
## 4.15 小结

- **反转变换**：$s = 255 - r$，增强暗部细节；
- **线性变换**：$s = a \cdot r + b$，调整对比度和亮度；`cv2.convertScaleAbs` 自动取绝对值并饱和截断，无需手工 `np.clip`；
- **对数变换**：$s = c \cdot \log(1 + r)$，扩展暗部细节，曲线形状固定；
- **Gamma 校正**：$s = c \cdot r^\gamma$，$\gamma < 1$ 提亮暗部、$\gamma > 1$ 压暗；由于 $\gamma$ 可连续取值，比对数变换更灵活、可调；LUT 加速；
- **分段线性变换**：对比度拉伸（适合灰度集中的低对比度图）、灰度级分层（Binary Slicing / Keep-background Slicing）；
- **比特平面**：把每个像素的 8 位二进制按位拆开，每一位单独组成一张二值图；第 `i` 位每 `2^i` 个灰度级翻转一次；第 7 位等价于“是否 ≥128”；高位权重大、翻转少，包含主要视觉信息，低位权重小、翻转频繁，主要是噪声；
- **直方图**：`cv2.calcHist` 计算，`plt.plot` 绘制；用硬币图、指纹图、grayLena 对比，可以看出不同图像的灰度分布形态差别很大；
- **直方图均衡化**：`cv2.equalizeHist` 增强对比度；
- **直方图匹配**：`calcHist` + `LUT` 实现；匹配是单调映射，只能让 CDF 接近，不能让直方图完全相同；
- **CLAHE**：分块均衡化 + 直方图裁剪 + 双线性插值；`cv2.createCLAHE` 比全局均衡化更温和，能在增强局部对比度的同时抑制噪声放大；`clipLimit` 控制对比度限制，`tileGridSize` 控制分块大小；
- **彩色图像均衡化**：不能直接对 RGB 三通道分别均衡化，否则会偏色；应转到 YCrCb 或 Lab，只对 Y/L 通道均衡化；用颜色丰富 + 亮度有问题的图演示最直观；
- **2D 直方图**：H-S 平面联合直方图；显示时用 `log(1+count)` 压缩高值、抬高低值；`np.log1p` 用自然对数 e；
- **反向投影**：`cv2.calcBackProject` 用于目标定位；
- **噪声模型**：提出噪声模型是为了回答“噪声是什么、该用什么方法去、去得怎么样”三个问题；不同噪声对应不同方法：高斯用 NLM / 双边滤波，椒盐用中值滤波，泊松用方差稳定变换，乘性用同态滤波；
- **去噪**：非局部均值、双边滤波、中值滤波；
- **去噪评估**：PSNR 看像素误差（越大越好，dB），SSIM 看亮度、对比度、结构的相似度（越接近 1 越好）；两者结合才能判断去噪是真正变好还是只是变糊；PSNR 需要干净参考图，真实场景没有参考时不能直接算，详见附录 A；
- **图像修复**：`cv2.inpaint`，`INPAINT_TELEA`、`INPAINT_NS`。

**核心原则**：

1. 每一处讲解都配可运行代码，每一段代码都显示结果；
2. 每个代码块都**完全自包含**——自己导入库、自己读取图像（或自建测试数据），不依赖其他任何小节；
3. 这样做的好处是：你可以在任意小节单独运行代码，也方便复习某一节时不用从头跑一遍；
4. **变量名与术语分离**：图像处理术语（灰度、直方图、均衡化）不与变量名混用，图像数组统一用 `img_xxx` 前缀；
5. **示例图要与概念匹配**：反转变换、比特平面用 `fingerprint.png`；灰度变换、Gamma 校正、直方图均衡化、CLAHE 用 `bone.jpg`；灰度直方图用 `coin.jpg`、`fingerprint.png`、`grayLena.bmp` 对比；直方图匹配用 `bone.jpg` -> `fingerprint.png`；图像修复用 `coin.jpg`；彩色图像均衡化用 `darkcolorful.png`；2D 直方图与反向投影用 `portraitGrass.png`；去噪、去模糊用 `fingerprint.png` 自行加噪/加模糊；对比度拉伸、灰度级分层用**自制测试图**，因为自制图的灰度分区更明确，效果更直观；
6. **常见坑就地提醒**：每一节末尾都用 `> **常见坑**` 列出本节最容易犯的错误，并给出原因和解决方式，不再单独设“常见坑”汇总章节。

---
# 附录 A：PSNR 评估降噪的适用边界

## A.1 PSNR 为何能评估降噪

PSNR 能用来评估降噪，根本原因是：**它直接衡量“降噪后图像和干净图像之间的像素误差”，而“降噪”的目标就是让处理后的图像尽量接近干净图像**。

降噪的输入是“含噪图”，输出是“去噪图”。理想情况下，去噪图应该尽量接近“干净图”。含噪图和干净图差距大 → 降噪效果差；去噪图和干净图差距小 → 降噪效果好。所以，只要有一个指标能衡量“去噪图和干净图的差距”，就能用来评估降噪。PSNR 就是这样的指标。

PSNR 由 MSE（均方误差）导出：

$$MSE = \frac{1}{MN}\sum_{i=1}^{M}\sum_{j=1}^{N}(I_{clean}(i,j) - I_{denoised}(i,j))^2$$

$$PSNR = 10 \cdot \log_{10}\left(\frac{255^2}{MSE}\right)$$

- MSE 越小，说明去噪图和干净图越接近；
- PSNR 越大，说明失真越小，降噪效果越好。

PSNR 适合评估降噪的原因：

- **它直接对应“降噪目标”**：降噪的目标就是“让输出接近干净图”，PSNR 衡量的正是“输出和干净图有多接近”；
- **它可量化、可比较**：不同降噪方法、不同参数，都能在同一张干净图上加同样的噪声、再去噪、算 PSNR，直接比较；
- **它对噪声抑制敏感**：含噪图的 PSNR 通常只有 20~25 dB，去噪后升到 28~38 dB，升幅越大，说明噪声被抑制得越彻底；
- **它和 MSE 一一对应**：PSNR 是 MSE 的单调递减函数，MSE 越小，PSNR 越大。

PSNR 的局限：只看像素误差，不关心结构。去噪后整体模糊时，PSNR 可能不低，但结构已经坏了；所以评估降噪时通常 PSNR 和 SSIM 一起看。

## A.2 真实场景没有干净图怎么办

PSNR 的公式必须要有 `I_clean`，也就是“干净图”。但现实里：

- 你拍到的照片本身就是含噪的；
- 你不知道“如果没噪声，这张图应该长什么样”；
- 所以**真实场景中无法直接算 PSNR**。

那为什么论文和实验里还在用 PSNR？因为实验里可以**人为构造“已知干净图”**：

1. 找一张干净图 `I_clean`（比如指纹图、Lena 图）；
2. 按噪声模型加噪，得到 `I_noisy`；
3. 用降噪方法处理 `I_noisy`，得到 `I_denoised`；
4. 用 `I_clean` 和 `I_denoised` 算 PSNR。

这样就有了“参照标准”，可以定量比较不同方法、不同参数。**这是实验评估，不是真实场景评估。**

## A.3 实践中什么时候能用 PSNR

**（1）有干净参考图的场景**

- **医学影像**：同一部位可能有多帧、多次曝光，可以选一帧作为参考；
- **遥感图像**：同一区域多次拍摄，可以选最清晰的一帧作为参考；
- **工业检测**：可以拍一张“无缺陷、低噪声”的参考图；
- **图像修复 / 超分辨率**：有高分辨率原图作为参考。

**（2）合成数据 / 仿真数据**

- 用已知干净图加已知噪声，构造测试集；
- 用来比较不同降噪算法、不同参数；
- 这是论文和工程中最常用的方式。

**（3）有“高质量参考”的替代品**

- 多帧平均：多张含噪图平均后，噪声被抑制，可以作为“准干净图”；
- 长曝光 / 低 ISO 拍一张参考图；
- 用这些“准干净图”作为参考，PSNR 仍可近似使用。

## A.4 不能用 PSNR 时怎么办

**（1）无参考指标**

- **噪声水平估计**：估计去噪后还剩多少噪声；
- **清晰度指标**：Laplacian 方差、Tenengrad，看边缘是否被磨平；
- **自然图像统计**：NIQE、BRISQUE，判断图像是否“像自然图像”；
- **盲去噪评估**：用去噪前后的噪声估计变化来评估。

**（2）人眼主观评价**

- 让多个人看原图、去噪图，打分或排序；
- 常用 MOS（平均主观意见分）；
- 缺点是费时费力，不同人标准不同。

**（3）任务驱动评估**

- 去噪后做指纹识别，看识别率；
- 去噪后做目标检测，看 mAP；
- 去噪后做分割，看 IoU；
- 任务指标提升，说明去噪对任务有帮助。

**（4）多帧 / 多视角参考**

- 如果能拍到多帧，可以用多帧平均作为参考；
- 或者用不同视角、不同曝光，构造“准干净图”。

## A.5 实践中的常见做法

**（1）论文 / 算法开发**

- 用合成数据（干净图 + 已知噪声）算 PSNR、SSIM；
- 这是标准做法，便于比较不同算法。

**（2）工程部署**

- 没有干净图时，用无参考指标 + 任务指标；
- 有干净图时，PSNR 仍然可用；
- 实际工程中，往往 **PSNR 只是参考之一**，最终看任务效果和主观感受。

**（3）医学影像 / 遥感**

- 有时能获得多次扫描，选一帧作为参考；
- 没有参考时，用医生诊断准确率、地物分类精度等任务指标。

## A.6 一句话总结

- **PSNR 是“有参考时的指标”**，不是“万能指标”；
- 实践中能用 PSNR 评估降噪，**前提是能拿到干净参考图**：合成数据、多帧平均、医学影像多次扫描、遥感多次拍摄等场景都可以用；
- 单张含噪照片、老照片修复、单次扫描等没有参考的场景，**不能直接算 PSNR**，要用无参考指标、人眼主观评价或任务驱动评估；
- 论文和算法开发中，PSNR 是标准指标，因为可以人为构造干净图 + 噪声；
- 工程部署中，PSNR 通常只是参考之一，最终看任务效果和主观感受。